 <b><i> data_process_paper </i></b>

Created by Eduardo Alastrue de Asenjo on 2026-03-09
   - Purpose: process data needed for analysis

# Init

Load modules

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
import matplotlib.path as mpath
import seaborn as sns
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from shapely.geometry import Polygon
import re
import math as math
import netCDF4 as nc
from scipy import stats
from random import choices
import copy
import dask
import dask.array as da
import xarray as xr
import pandas as pd
import datetime
import cftime
from tqdm import tqdm
import glob
import pickle
import cdo 
import statsmodels.api as sm 
import xclim as xc
import eofs
import xesmf as xe
import nc_time_axis
import cf_xarray as cfxr
import regionmask
import warnings
import json
warnings.filterwarnings("ignore", category=RuntimeWarning)

run local dask cluster

In [ ]:
from dask.distributed import Client
client = Client(memory_limit=None, threads_per_worker=1, n_workers=18)

load functions

In [ ]:
def weighted_mon_to_year_mean(ds, var):
    """
    weight by days in each month when doing annual mean from monthly values in xarray
    taken from https://ncar.github.io/esds/posts/2021/yearly-averages-xarray/
    """
    month_length = ds.time.dt.days_in_month # Determine the month length

    wgts = month_length.groupby("time.year") / month_length.groupby("time.year").sum() # Calculate the weights

    np.testing.assert_allclose(wgts.groupby("time.year").sum(xr.ALL_DIMS), 1.0) # Make sure the weights in each year add up to 1

    obs = ds[var] # Subset our dataset for our variable

    cond = obs.isnull() # Setup our masking for nan values
    ones = xr.where(cond, 0.0, 1.0)
    
    obs_sum = (obs * wgts).resample(time="YS").sum(dim="time") # Calculate the numerator
    ones_out = (ones * wgts).resample(time="YS").sum(dim="time") # Calculate the denominator

    return obs_sum / ones_out # Return the weighted average

In [ ]:
def weighted_area_lat(ds):
    """
    Calculate the area-weighted temperature over its domain. 
    In a regular latitude/ longitude grid the grid cell area decreases towards the pole.
    We can use the cosine of the latitude as proxy for the grid cell area.
    IMPORTANT: after applying function, do .mean('lat') before 'lon'
    Taken from https://docs.xarray.dev/en/stable/examples/area_weighted_temperature.html
    """
    weights = np.cos(np.deg2rad(ds.lat))
    weights.name = "weights"

    return ds.weighted(weights)

In [ ]:
def reset_djf_time(ds):
    '''
    Function to change xarray dates back to the January of a given winter, after 
    .resample(time='QS-DEC').mean(dim='time'), & .sel(time=seas.time.dt.season=='DJF')
    '''
    def change_month(date):
        return cftime.DatetimeProlepticGregorian(date.year + 1, 1, date.day)
        return date
    new_time = [change_month(date) for date in ds['time'].values]
    ds['time'] = new_time
    return ds

# MPI-ESM SSP+hosing

In [ ]:
# note that this data is not cmorised, but in the format of the direct output from MPI-ESM1.2-LR

In [ ]:
ssphos_lst = []
for exp in ["ssp126", "ssp245", "ssp370"]:
    for forc in ["grcneg01Sv", "grc01Sv", "grc03Sv", "grc05Sv", "grclinneg02Sv", "grclin02Sv", "grclin06Sv", "grclin10Sv"]:
        ssphos_lst.append(f"{exp}_{forc}") 
realiz_lst = ["r1i1p1f1", "r2i1p1f1", "r3i1p1f1"]

## AMOC

In [ ]:
for exp in ssphos_lst:
    for rea in realiz_lst: 
        outpath = f"/work/uo1075/m300817/teu_amoc/data/{exp}_{rea}/" 
        if not os.path.exists(outpath):
            os.makedirs(outpath)
        if not os.path.isfile(outpath+f"{exp}_{rea}_amoc26_yr.nc"):
            msftmz = xr.open_mfdataset(f"/work/uo1075/m300817/hosing/mpiesm-1.2.01p7-passivesalt-hosing/experiments/{exp}_{rea}/outdata/mpiom/*_mpiom_data_moc_mm_*.nc"
                                       , use_cftime=True, parallel=True)
            atlmoc = msftmz.drop_vars([var for var in msftmz.data_vars if var != "atlantic_moc"])
            amoc = (atlmoc.sel(lat=26.5, drop=True).sel(lon=0, drop=True)/(1025.0 * 10**6)).compute()
            max_dep_idx= amoc.idxmax(dim='depth_2') #select depths with maximum msftmz
            amoc26_yr = weighted_mon_to_year_mean(amoc.sel(depth_2=max_dep_idx.atlantic_moc, drop=True), "atlantic_moc")
            amoc26_yr.to_netcdf(outpath+f"{exp}_{rea}_amoc26_yr.nc")

In [ ]:
ssphos_amoc_yr_array = []
for exp in ssphos_lst:
    rea_array =[]
    for rea in realiz_lst: 
        rea_array.append(xr.open_dataarray(f"//work/uo1075/m300817/teu_amoc/data/{exp}_{rea}/{exp}_{rea}_amoc26_yr.nc",
                                           use_cftime=True).to_dataset(name='amoc').assign_coords(realiz=rea))
    ssphos_amoc_yr_array.append(xr.concat(rea_array, dim='realiz').assign_coords(exper=exp))
ssphos_amoc_yr = xr.concat(ssphos_amoc_yr_array, dim='exper')
outpath = f"/work/uo1075/m300817/teu_amoc/data/ssphos/" 
ssphos_amoc_yr.to_netcdf(outpath+"ssphos_amoc_yr.nc")

## SATs

In [ ]:
ssphos_lst = []
for exp in ["ssp126", "ssp245", "ssp370"]:
    for forc in ["grcneg01Sv","grc01Sv", "grc03Sv", "grc05Sv", "grclinneg02Sv", "grclin02Sv", "grclin06Sv", "grclin10Sv"]:
        ssphos_lst.append(f"{exp}_{forc}") 
realiz_lst = ["r1i1p1f1", "r2i1p1f1", "r3i1p1f1"]

preprocess and save

In [ ]:
# tas grib to nc 
cdo.setCdo("/sw/spack-levante/cdo-1.9.10-j5frmz/bin/cdo")
for exp in ssphos_lst:
    for rea in realiz_lst: 
        inpath  = f"/work/uo1075/m300817/hosing/mpiesm-1.2.01p7-passivesalt-hosing/experiments/{exp}_{rea}/outdata/echam6/"
        outpath = f"/work/uo1075/m300817/teu_amoc/data/{exp}_{rea}/" 
        if not os.path.exists(outpath):
            os.makedirs(outpath)
        for year in range (2015, 2101):
            outfile=f"{exp}_{rea}_echam6_echam_{year}.nc"
            if not os.path.isfile(outpath+outfile):
                cdo.copy(input = f"-selcode,167 {inpath}{exp}_{rea}_echam6_echam_{year}.grb", output=outpath+outfile, options = '-f nc -t echam6 ')

In [ ]:
# Load resulting file
ssphos_tas_mon_array = []
for exp in ssphos_lst:
    rea_array =[]
    for rea in realiz_lst: 
        ifiles  = f"/work/uo1075/m300817/teu_amoc/data/{exp}_{rea}/{exp}_{rea}_echam6_echam_*.nc"
        tas = xr.open_mfdataset(ifiles, use_cftime=True).rename({'temp2': 'tas'})
        tas = tas.assign_coords(time=xr.cftime_range(start="2015-01-01", end="2100-12-31", freq="M", calendar="proleptic_gregorian"))
        tas.coords['lon'] = (tas.coords['lon'] + 180) % 360 - 180 # center longitudes at 0
        tas = tas.sortby(tas.lon)
        rea_array.append(tas.assign_coords(realiz=rea))
    ssphos_tas_mon_array.append(xr.concat(rea_array, dim='realiz').assign_coords(exper=exp))
ssphos_tas_mon =  xr.concat(ssphos_tas_mon_array, dim='exper')
ssphos_tas_yr = weighted_mon_to_year_mean(ssphos_tas_mon, "tas")

In [ ]:
outpath = f"/work/uo1075/m300817/teu_amoc/data/ssphos/" 
ssphos_tas_mon.to_netcdf(outpath+"ssphos_tas_mon.nc")
ssphos_tas_yr.to_netcdf(outpath+"ssphos_tas_yr.nc")

seasons

In [ ]:
ssphos_tas_sea = ssphos_tas_mon.resample(time='QS-DEC').mean(dim='time')
ssphos_tas_djf = ssphos_tas_sea.sel(time=ssphos_tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1)) # remove year 2101, only Dec 2100
ssphos_tas_djf = reset_djf_time(ssphos_tas_djf)
outpath = f"/work/uo1075/m300817/teu_amoc/data/ssphos/" 
ssphos_tas_djf.to_netcdf(outpath+"ssphos_tas_djf.nc")

In [ ]:
ssphos_tas_sea = ssphos_tas_mon.resample(time='QS').mean(dim='time')
ssphos_tas_jja = ssphos_tas_sea.sel(time=ssphos_tas_sea.time.dt.season=='JJA')
outpath = f"/work/uo1075/m300817/teu_amoc/data/ssphos/" 
ssphos_tas_jja.to_netcdf(outpath+"ssphos_tas_jja.nc")

masked regions

In [ ]:
mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(ssphos_tas_yr)  # Europe
mask_neu = mask_eur == 16
mask_wce = mask_eur == 17
mask_eeu = mask_eur == 18
mask_med = mask_eur == 19
masks_eur_dict = {'NEU': mask_neu, 'WCE': mask_wce, 'EEU': mask_eeu, 'MED': mask_med}

mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(ssphos_tas_yr) # Natural earth masks
mask_ocean = regionmask.defined_regions.natural_earth_v5_0_0.ocean_basins_50.mask(ssphos_tas_yr)

In [ ]:
ssphos_tas_eur_mon = ssphos_tas_mon.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
ssphos_tas_eur_yr = ssphos_tas_yr.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
ssphos_tas_eur_djf = ssphos_tas_djf.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
ssphos_tas_eur_jja = ssphos_tas_jja.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)                                                       

In [ ]:
outpath = f"/work/uo1075/m300817/teu_amoc/data/ssphos/" 
ssphos_tas_eur_mon.to_netcdf(outpath+"ssphos_tas_eur_mon.nc")
ssphos_tas_eur_yr.to_netcdf(outpath+"ssphos_tas_eur_yr.nc")
ssphos_tas_eur_djf.to_netcdf(outpath+"ssphos_tas_eur_djf.nc")
ssphos_tas_eur_jja.to_netcdf(outpath+"ssphos_tas_eur_jja.nc")

## daily tas

preprocess and save

In [ ]:
# daily grib to nc 
cdo.setCdo("/sw/spack-levante/cdo-1.9.10-j5frmz/bin/cdo")
for exp in ['ssp245_grc1Sv']:
    for rea in ['r1i1p1f1']: 
        inpath  = f"/work/uo1075/m300817/hosing/mpiesm-1.2.01p7-passivesalt-hosing/experiments/{exp}_{rea}/outdata/echam6/"
        outpath = f"/work/uo1075/m300817/teu_amoc/data/{exp}_{rea}/" 
        for year in range (2015, 2299):
            outfile=f"{exp}_{rea}_echam6_echamdaily_{year}.nc"
            if not os.path.isfile(outpath+outfile):
                cdo.copy(input = f"{inpath}{exp}_{rea}_echam6_echamdaily_{year}.grb", 
                         output=outpath+outfile, options = '-f nc -t echam6 ')

In [ ]:
# Load and save resulting file
ssphos_tas_day_array = []
for exp in ['ssp245_grc1Sv']:
    rea_array =[]
    for rea in ['r1i1p1f1']: 
        ifiles  = f"/work/uo1075/m300817/teu_amoc/data/{exp}_{rea}/{exp}_{rea}_echam6_echamdaily_*.nc"
        tas = xr.open_mfdataset(ifiles, use_cftime=True)[["temp2"]]
        tas = tas.rename({'temp2': 'tas'})
        tas = tas.assign_coords(time=xr.cftime_range(start="2015-01-01", end="2298-12-31", freq="D", calendar="proleptic_gregorian"))
        tas.coords['lon'] = (tas.coords['lon'] + 180) % 360 - 180 # center longitudes at 0
        tas = tas.sortby(tas.lon)
        rea_array.append(tas.assign_coords(realiz=rea))
    ssphos_tas_day_array.append(xr.concat(rea_array, dim='realiz').assign_coords(exper=exp))
ssphos_tas_day =  xr.concat(ssphos_tas_day_array, dim='exper')
outpath = f"/work/uo1075/m300817/teu_amoc/data/ssphos/" 
ssphos_tas_day.to_netcdf(outpath+"ssphos_tas_day.nc")

Extreme indices

In [ ]:
# TMn year
tmn_year = xc.indices.tn_min(ssphos_tas_day.sel(exper='ssp245_grc1Sv').isel(realiz=0).tas, freq='YS')
tmn_year.to_netcdf(f"/work/uo1075/m300817/teu_amoc/data/ssphos/tmn_ssp245_grc1Sv_year.nc")

In [ ]:
# TMn month
tmn_mon = xc.indices.tn_min(ssphos_tas_day.sel(exper='ssp245_grc1Sv').isel(realiz=0).tas, freq='MS')
tmn_mon.to_netcdf(f"/work/uo1075/m300817/teu_amoc/data/ssphos/tmn_ssp245_grc1Sv_mon.nc")

# MPI-GE CMIP6, no hosing (cmor)

In [ ]:
ge_paths= {'historical'    : "/pool/data/CMIP6/data/CMIP/MPI-M/MPI-ESM1-2-LR/historical",
           'piControl'     : "/pool/data/CMIP6/data/CMIP/MPI-M/MPI-ESM1-2-LR/piControl/r1i1p1f1",
           'ssp126'        : '/pool/data/CMIP6/data/ScenarioMIP/MPI-M/MPI-ESM1-2-LR/ssp126',
           'ssp245'        : '/pool/data/CMIP6/data/ScenarioMIP/MPI-M/MPI-ESM1-2-LR/ssp245',
           'ssp370'        : '/pool/data/CMIP6/data/ScenarioMIP/MPI-M/MPI-ESM1-2-LR/ssp370',
          }  

## AMOC

preprocess and save (can skip after run once if files still exist)

In [ ]:
array = []
for i in range(1,51):
    rea = "r"+str(i)+"i1p1f1"
    if os.path.isdir(f'{ge_paths["historical"]}/{rea}'):
        infiles = glob.glob(f'{ge_paths["historical"]}/{rea}/Omon/msftmz/gn/**/*.nc', recursive=True)
        array.append(xr.open_mfdataset(infiles, use_cftime=True, data_vars="minimal", coords="minimal", chunks={'time': 1980}, parallel=True, compat="override").assign_coords(realiz=rea)) # compat="override"
his_msftmz_mon = xr.concat(array, dim='realiz')

In [ ]:
# AMOC strength 26.5N in Sv
his_amoc_yr = weighted_mon_to_year_mean(his_msftmz_mon.sel(basin=1, drop=True).sel(lat=26.5, drop=True), "msftmz")
max_lev_idx= his_amoc_yr.idxmax(dim='lev') #select depths with maximum msftmz
his_amoc_yr = (his_amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
his_amoc_yr = his_amoc_yr.load() 
his_amoc_yr.to_netcdf(outpath+"his_amoc26_yr.nc")

In [ ]:
# AMOC strength 40N in Sv
his_amoc_yr = weighted_mon_to_year_mean(his_msftmz_mon.sel(basin=1, drop=True).sel(lat=40, method='nearest', drop=True), "msftmz")
max_lev_idx= his_amoc_yr.idxmax(dim='lev') #select depths with maximum msftmz
his_amoc_yr = (his_amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
his_amoc_yr = his_amoc_yr.load() 
his_amoc_yr.to_netcdf(outpath+"his_amoc40_yr.nc")

In [ ]:
array_ssp = []
for exp in ['ssp126','ssp245', 'ssp370', 'ssp585']:      
    array_rea = []
    for i in range(1,51):
        rea = "r"+str(i)+"i1p1f1"
        if os.path.isdir(f'{ge_paths[exp]}/{rea}'):
            infiles = glob.glob(f'{ge_paths[exp]}/{rea}/Omon/msftmz/gn/**/*.nc', recursive=True)
            array_rea.append(xr.open_mfdataset(infiles, use_cftime=True, data_vars="minimal", coords="minimal", chunks={'time': 1980}, parallel=True, compat="override").assign_coords(realiz=rea)) # compat="override"
    array_ssp.append(xr.concat(array_rea, dim='realiz').assign_coords(scenar=exp))
ssp_msftmz_mon = xr.concat(array_ssp, dim='scenar')

In [ ]:
# AMOC strength 26.5N in Sv
ssp_amoc_yr = weighted_mon_to_year_mean(ssp_msftmz_mon.sel(basin=1, drop=True).sel(lat=26.5, drop=True), "msftmz")
max_lev_idx= ssp_amoc_yr.idxmax(dim='lev') #select depths with maximum msftmz
ssp_amoc_yr = (ssp_amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
ssp_amoc_yr = ssp_amoc_yr.load() 
ssp_amoc_yr.to_netcdf(outpath+"ssp_amoc26_yr.nc")

In [ ]:
array_ssp = []
for exp in ['ssp126','ssp245', 'ssp370', 'ssp585']:      
    array_rea = []
    for i in range(1,51):
        rea = "r"+str(i)+"i1p1f1"
        if os.path.isdir(f'{ge_paths[exp]}/{rea}'):
            infiles = glob.glob(f'{ge_paths[exp]}/{rea}/Omon/msftmz/gn/**/*.nc', recursive=True)
            array_rea.append(xr.open_mfdataset(infiles, use_cftime=True, data_vars="minimal", coords="minimal", chunks={'time': 1980}, parallel=True, compat="override").assign_coords(realiz=rea)) # compat="override"
    array_ssp.append(xr.concat(array_rea, dim='realiz').assign_coords(scenar=exp))
ssp_msftmz_mon = xr.concat(array_ssp, dim='scenar')

In [ ]:
# AMOC strength 40N in Sv
ssp_amoc_yr = weighted_mon_to_year_mean(ssp_msftmz_mon.sel(basin=1, drop=True).sel(lat=40, method='nearest', drop=True), "msftmz")
max_lev_idx= ssp_amoc_yr.idxmax(dim='lev') #select depths with maximum msftmz
ssp_amoc_yr = (ssp_amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
ssp_amoc_yr = ssp_amoc_yr.load() 
ssp_amoc_yr.to_netcdf(outpath+"ssp_amoc40_yr.nc")

## SAT

In [ ]:
array = []
for i in range(1,51):
    rea = "r"+str(i)+"i1p1f1"
    if os.path.isdir(f'{ge_paths["historical"]}/{rea}'):
        infiles = glob.glob(f'{ge_paths["historical"]}/{rea}/Amon/tas/gn/**/*.nc', recursive=True)
        array.append(xr.open_mfdataset(infiles, use_cftime=True, data_vars="minimal", coords="minimal", chunks={'time': 1980}, parallel=True, compat="override").assign_coords(realiz=rea)) # compat="override"
his_tas_mon = xr.concat(array, dim='realiz')
his_tas_yr = weighted_mon_to_year_mean(his_tas_mon, "tas")
his_tas_yr = his_tas_yr.load()
his_tas_yr.to_netcdf(outpath+"his_tas_yr.nc")

In [ ]:
his_tas_mon.to_netcdf(outpath+"his_tas_mon.nc")

In [ ]:
array_ssp = []
for exp in ['ssp126','ssp245','ssp370']:   
    array_rea = []
    for i in range(1,51):
        rea = "r"+str(i)+"i1p1f1"
        if os.path.isdir(f'{ge_paths[exp]}/{rea}'):
            infiles = glob.glob(f'{ge_paths[exp]}/{rea}/Amon/tas/gn/**/*.nc', recursive=True)
            array_rea.append(xr.open_mfdataset(infiles, use_cftime=True, data_vars="minimal", coords="minimal", chunks={'time': 1980}, parallel=True, compat="override").assign_coords(realiz=rea)) # compat="override"
    array_ssp.append(xr.concat(array_rea, dim='realiz').assign_coords(scenar=exp))
ssp_tas_mon = xr.concat(array_ssp, dim='scenar')
ssp_tas_mon = ssp_tas_mon.load()
ssp_tas_mon.to_netcdf(outpath+"ssp_tas_mon.nc")

ssp_tas_yr = weighted_mon_to_year_mean(ssp_tas_mon, "tas")
ssp_tas_yr = ssp_tas_yr.load()
ssp_tas_yr.to_netcdf(outpath+"ssp_tas_yr.nc")

seasons

In [ ]:
his_tas_sea = his_tas_mon.resample(time='QS-DEC').mean(dim='time')
his_tas_djf = his_tas_sea.sel(time=his_tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1)) # remove year 2015, only Dec 2014
his_tas_djf = reset_djf_time(his_tas_djf)

ssp_tas_sea = ssp_tas_mon.resample(time='QS-DEC').mean(dim='time')
ssp_tas_djf = ssp_tas_sea.sel(time=ssp_tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1)) # remove year 2101, only Dec 2100
ssp_tas_djf = reset_djf_time(ssp_tas_djf)

his_tas_djf.to_netcdf(outpath+"his_tas_djf.nc")
ssp_tas_djf.to_netcdf(outpath+"ssp_tas_djf.nc")

In [ ]:
his_tas_sea = his_tas_mon.resample(time='QS').mean(dim='time')
his_tas_jja = his_tas_sea.sel(time=his_tas_sea.time.dt.season=='JJA')

ssp_tas_sea = ssp_tas_mon.resample(time='QS').mean(dim='time')
ssp_tas_jja = ssp_tas_sea.sel(time=ssp_tas_sea.time.dt.season=='JJA')

his_tas_jja.to_netcdf(outpath+"his_tas_jja.nc")
ssp_tas_jja.to_netcdf(outpath+"ssp_tas_jja.nc")

masked regions

In [ ]:
mask_ge_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(his_tas_yr)  # Europe
mask_ge_neu = mask_ge_eur == 16
mask_ge_wce = mask_ge_eur == 17
mask_ge_eeu = mask_ge_eur == 18
mask_ge_med = mask_ge_eur == 19
masks_ge_eur_dict = {'NEU': mask_ge_neu, 'WCE': mask_ge_wce, 'EEU': mask_ge_eeu, 'MED': mask_ge_med}

mask_ge_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(his_tas_yr) # Natural earth masks
mask_ge_ocean = regionmask.defined_regions.natural_earth_v5_0_0.ocean_basins_50.mask(his_tas_yr)

In [ ]:
his_tas_eur_yr = his_tas_yr.where(mask_ge_eur>15, drop=True).where(mask_ge_land==0, drop=True)
his_tas_eur_djf = his_tas_djf.where(mask_ge_eur>15, drop=True).where(mask_ge_land==0, drop=True)
his_tas_eur_jja = his_tas_jja.where(mask_ge_eur>15, drop=True).where(mask_ge_land==0, drop=True)
his_tas_eur_yr.to_netcdf(outpath+"his_tas_eur_yr.nc")
his_tas_eur_djf.to_netcdf(outpath+"his_tas_eur_djf.nc")
his_tas_eur_jja.to_netcdf(outpath+"his_tas_eur_jja.nc")


ssp_tas_eur_yr = ssp_tas_yr.where(mask_ge_eur>15, drop=True).where(mask_ge_land==0, drop=True)
ssp_tas_eur_djf = ssp_tas_djf.where(mask_ge_eur>15, drop=True).where(mask_ge_land==0, drop=True)
ssp_tas_eur_jja = ssp_tas_jja.where(mask_ge_eur>15, drop=True).where(mask_ge_land==0, drop=True)
ssp_tas_eur_yr.to_netcdf(outpath+"ssp_tas_eur_yr.nc")
ssp_tas_eur_djf.to_netcdf(outpath+"ssp_tas_eur_djf.nc")
ssp_tas_eur_jja.to_netcdf(outpath+"ssp_tas_eur_jja.nc")

daily data

In [ ]:
array_year = []
array_mon  = []
for i in range(1,51):
    rea = "r"+str(i)+"i1p1f1"
    if os.path.isdir(f'{ge_paths["historical"]}/{rea}'):
        infiles = glob.glob(f'{ge_paths["historical"]}/{rea}/day/tas/gn/**/*.nc', recursive=True)
        his_tas_day_rea = xr.open_mfdataset(infiles, use_cftime=True, data_vars="minimal", coords="minimal", chunks={'time': 1980}, parallel=True, compat="override").assign_coords(realiz=rea)
        array_year.append(xc.indices.tn_min(his_tas_day_rea.tas, freq='YS')) 
        array_mon.append(xc.indices.tn_min(his_tas_day_rea.tas, freq='MS')) 
his_tmn_year = xr.concat(array_year, dim='realiz')
his_tmn_year.to_netcdf(outpath+"his_tmn_year.nc")
his_tmn_mon = xr.concat(array_mon, dim='realiz')
his_tmn_mon.to_netcdf(outpath+"his_tmn_mon.nc")

In [ ]:
array_ssp_year  = []
array_ssp_mon  = []
for exp in ['ssp126','ssp245','ssp370','ssp585']:   
    array_rea_year = []
    array_rea_mon = []
    for i in range(1,51):
        rea = "r"+str(i)+"i1p1f1"
        if os.path.isdir(f'{ge_paths[exp]}/{rea}'):
            infiles = glob.glob(f'{ge_paths[exp]}/{rea}/day/tas/gn/**/*.nc', recursive=True)
            tas_day_rea = xr.open_mfdataset(infiles, use_cftime=True, data_vars="minimal", coords="minimal", chunks={'time': 1980}, parallel=True, compat="override").assign_coords(realiz=rea)
            array_rea_year.append(xc.indices.tn_min(tas_day_rea.tas, freq='YS')) 
            array_rea_mon.append(xc.indices.tn_min(tas_day_rea.tas, freq='MS')) 
    array_ssp_year.append(xr.concat(array_rea_year, dim='realiz').assign_coords(scenar=exp))
    array_ssp_mon.append(xr.concat(array_rea_mon, dim='realiz').assign_coords(scenar=exp))
ssp_tmn_year = xr.concat(array_ssp_year, dim='scenar')
ssp_tmn_mon = xr.concat(array_ssp_mon, dim='scenar')
ssp_tmn_year.to_netcdf(outpath+"ssp_tmn_year.nc")
ssp_tmn_mon.to_netcdf(outpath+"ssp_tmn_mon.nc")

# Other CMIP6 models

In [ ]:
# docu: https://freva-clint.github.io/freva/Freva.html#searching-for-data , https://docs.dkrz.de/doc/dataservices/finding_and_accessing_data/freva/index.html
import freva
freva.config(config_file='/work/ch1187/clint/freva-dev/freva/evaluation_system.conf')

## AMOC msftmz

### ssp126

In [ ]:
ssp126_facet = freva.facet_search(experiment="ssp126", product="scenariomip", project="cmip6", variable="msftmz", time="2015 to 2100", time_select="strict")

In [ ]:
ssp126_msftmz_cmip6 = {}
for model in ssp126_facet['model']:
    if not model=='mpi-esm1-2-lr': # already processed before
        realiz_facet = freva.facet_search(experiment="ssp126", model=model, product="scenariomip", project="cmip6", variable="msftmz", time="2015 to 2100", time_select="strict")
        ssp126_msftmz_cmip6[model] = {}
        for rea in realiz_facet['ensemble']:
            files = freva.databrowser(
                ensemble=rea,experiment="ssp126", model=model, product="scenariomip", project="cmip6", 
                variable="msftmz", time="2015 to 2100", time_select="strict")
            ssp126_msftmz_cmip6[model][rea] = xr.open_mfdataset(files, parallel=True, use_cftime=True)

In [ ]:
# Check time steps
for model in ssp126_msftmz_cmip6.keys():
    for rea in ssp126_msftmz_cmip6[model].keys():
        msftmz = ssp126_msftmz_cmip6[model][rea]
        print(f"{model},{rea} "+str(msftmz.isel(time=0).time.dt.year.values)+", "+str(msftmz.isel(time=-1).time.dt.year.values)+", "+str(len(msftmz.time.dt.year.values)))

In [ ]:
for model in ssp126_msftmz_cmip6.keys():
    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/{model}/"
    if not os.path.exists(outpath):
        os.makedirs(outpath)
    basin = 1
    if not model == 'mpi-esm1-2-hr': # only mpi models have basin = 0 for the atlantic
        basin = 0
    for rea in ssp126_msftmz_cmip6[model].keys():
        msftmz = ssp126_msftmz_cmip6[model][rea]
        if len(msftmz.time.dt.year.values)==1032:
            if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
                amoc_yr = weighted_mon_to_year_mean(msftmz.sel(basin=basin, drop=True).sel(lat=26.5, method='nearest', drop=True), "msftmz")
                max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftmz
                amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
                amoc_yr = amoc_yr.load() 
                amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
# dictionary realiz for each model
cmip6_ssp126_amoc_dict = {}
for model in ssp126_msftmz_cmip6.keys():
    cmip6_ssp126_amoc_dict[model] = []
    for rea in ssp126_msftmz_cmip6[model].keys():
        msftmz = ssp126_msftmz_cmip6[model][rea]
        if len(msftmz.time.dt.year.values)==1032:
            cmip6_ssp126_amoc_dict[model].append(rea)
    if not cmip6_ssp126_amoc_dict[model]: #remove key from dict if realiz list empty 
        del cmip6_ssp126_amoc_dict[model]

with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/cmip6_ssp126_amoc_dict.json", 'w') as json_file:
    json.dump(cmip6_ssp126_amoc_dict, json_file)

In [ ]:
# separately process cesm2 member, even if not complete period
model = 'cesm2'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
basin = 0
for rea in ssp126_msftmz_cmip6[model].keys():
    msftmz = ssp126_msftmz_cmip6[model][rea]
    if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
        amoc_yr = weighted_mon_to_year_mean(msftmz.sel(basin=basin, drop=True).sel(lat=26.5, method='nearest', drop=True), "msftmz")
        max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftmz
        amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
        amoc_yr = amoc_yr.load() 
        amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

### ssp245

In [ ]:
ssp245_facet = freva.facet_search(experiment="ssp245", product="scenariomip", project="cmip6", variable="msftmz", time="2015 to 2100", time_select="strict")

In [ ]:
ssp245_msftmz_cmip6 = {}
for model in ssp245_facet['model']:
    if not model=='mpi-esm1-2-lr': # already processed before
        realiz_facet = freva.facet_search(experiment="ssp245", model=model, product="scenariomip", project="cmip6", variable="msftmz", time="2015 to 2100", time_select="strict")
        ssp245_msftmz_cmip6[model] = {}
        for rea in realiz_facet['ensemble']:
            files = freva.databrowser(
                ensemble=rea,experiment="ssp245", model=model, product="scenariomip", project="cmip6", 
                variable="msftmz", time="2015 to 2100", time_select="strict")
            ssp245_msftmz_cmip6[model][rea] = xr.open_mfdataset(files, parallel=True, use_cftime=True)

In [ ]:
for model in ssp245_msftmz_cmip6.keys():
    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/{model}/"
    if not os.path.exists(outpath):
        os.makedirs(outpath)
    basin = 1
    if model not in ['mpi-esm1-2-hr','cas-esm2-0']: # only mpi models have basin = 0 for the atlantic
        basin = 0
    for rea in ssp245_msftmz_cmip6[model].keys():
        msftmz = ssp245_msftmz_cmip6[model][rea]
        if len(msftmz.time.dt.year.values)==1032:
            if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
                amoc_yr = weighted_mon_to_year_mean(msftmz.sel(basin=basin, drop=True).sel(lat=26.5, method='nearest', drop=True), "msftmz")
                max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftmz
                amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
                amoc_yr = amoc_yr.load() 
                amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
# dictionary realiz for each model
cmip6_ssp245_amoc_dict = {}
for model in ssp245_msftmz_cmip6.keys():
    cmip6_ssp245_amoc_dict[model] = []
    for rea in ssp245_msftmz_cmip6[model].keys():
        msftmz = ssp245_msftmz_cmip6[model][rea]
        if len(msftmz.time.dt.year.values)==1032:
            cmip6_ssp245_amoc_dict[model].append(rea)
    if not cmip6_ssp245_amoc_dict[model]: #remove key from dict if realiz list empty 
        del cmip6_ssp245_amoc_dict[model]

with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/cmip6_ssp245_amoc_dict.json", 'w') as json_file:
    json.dump(cmip6_ssp245_amoc_dict, json_file)

In [ ]:
# repeat for giss-e2-1-g from uploaded data
path = '/work/uo1075/m300817/teu_amoc/data/CMIP6/upload/giss-e2-1-g/'
basin = 0
for model in ['giss-e2-1-g']:
    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/{model}/"
    for rea in ['r1i1p1f2', 'r2i1p1f2', 'r3i1p1f2', 'r4i1p1f2', 'r5i1p1f2', 'r6i1p1f2', 'r7i1p1f2', 'r8i1p1f2', 'r9i1p1f2', 'r10i1p1f2']:
        msftmz = xr.open_mfdataset(path+"msftmz*ssp245*"+rea+"*", parallel=True, use_cftime=True)
        #if len(tas.time.dt.year.values)==1032:
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            amoc_yr = weighted_mon_to_year_mean(msftmz.sel(basin=basin, drop=True).sel(lat=26.5, method='nearest', drop=True), "msftmz")
            max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftmz
            amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

### ssp370

In [ ]:
ssp370_facet = freva.facet_search(experiment="ssp370", product="scenariomip", project="cmip6", variable="msftmz", time="2015 to 2100", time_select="strict")

In [ ]:
ssp370_msftmz_cmip6 = {}
for model in ssp370_facet['model']:
    if not model=='mpi-esm1-2-lr': # already processed before
        realiz_facet = freva.facet_search(experiment="ssp370", model=model, product="scenariomip", project="cmip6", variable="msftmz", time="2015 to 2100", time_select="strict")
        ssp370_msftmz_cmip6[model] = {}
        for rea in realiz_facet['ensemble']:
            files = freva.databrowser(
                ensemble=rea,experiment="ssp370", model=model, product="scenariomip", project="cmip6", 
                variable="msftmz", time="2015 to 2100", time_select="strict")
            ssp370_msftmz_cmip6[model][rea] = xr.open_mfdataset(files, parallel=True, use_cftime=True)

In [ ]:
for model in ssp370_msftmz_cmip6.keys():
    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/{model}/"
    if not os.path.exists(outpath):
        os.makedirs(outpath)
    basin = 1
    if model not in ['mpi-esm1-2-hr','cas-esm2-0', 'mpi-esm-1-2-ham']: # only mpi models have basin = 0 for the atlantic
        basin = 0
    for rea in ssp370_msftmz_cmip6[model].keys():
        msftmz = ssp370_msftmz_cmip6[model][rea]
        if len(msftmz.time.dt.year.values)==1032:
            if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
                amoc_yr = weighted_mon_to_year_mean(msftmz.sel(basin=basin, drop=True).sel(lat=26.5, method='nearest', drop=True), "msftmz")
                max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftmz
                amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
                amoc_yr = amoc_yr.load() 
                amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
# dictionary realiz for each model
cmip6_ssp370_amoc_dict = {}
for model in ssp370_msftmz_cmip6.keys():
    cmip6_ssp370_amoc_dict[model] = []
    for rea in ssp370_msftmz_cmip6[model].keys():
        msftmz = ssp370_msftmz_cmip6[model][rea]
        if len(msftmz.time.dt.year.values)==1032:
            cmip6_ssp370_amoc_dict[model].append(rea)
    if not cmip6_ssp370_amoc_dict[model]: #remove key from dict if realiz list empty 
        del cmip6_ssp370_amoc_dict[model]

with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/cmip6_ssp370_amoc_dict.json", 'w') as json_file:
    json.dump(cmip6_ssp370_amoc_dict, json_file)

In [ ]:
# repeat for cesm2 from uploaded data
path = '/work/uo1075/m300817/teu_amoc/data/CMIP6/upload/cesm2/'
basin = 0
for model in ['cesm2']:
    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/{model}/"
    for rea in ['r4i1p1f1', 'r10i1p1f1', 'r11i1p1f1']:
        msftmz = xr.open_mfdataset(path+"msftmz*ssp370*"+rea+"*", parallel=True, use_cftime=True)
        #if len(tas.time.dt.year.values)==1032:
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            amoc_yr = weighted_mon_to_year_mean(msftmz.sel(basin=basin, drop=True).sel(lat=26.5, method='nearest', drop=True), "msftmz")
            max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftmz
            amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

### historical

In [ ]:
historical_facet = freva.facet_search(experiment="historical", product="cmip", project="cmip6", variable="msftmz", time="1850 to 2014", time_select="strict")

In [ ]:
historical_msftmz_cmip6 = {}
for model in historical_facet['model']:
    if not model=='mpi-esm1-2-lr': # already processed before
        realiz_facet = freva.facet_search(model=model, experiment="historical", product="cmip", project="cmip6", variable="msftmz", time="1850 to 2014", time_select="strict")
        historical_msftmz_cmip6[model] = {}
        for rea in realiz_facet['ensemble']:
            files = freva.databrowser(
                ensemble=rea, model=model, experiment="historical", product="cmip", project="cmip6", 
                variable="msftmz", time="1850 to 2014", time_select="strict")
            historical_msftmz_cmip6[model][rea] = xr.open_mfdataset(files, parallel=True, use_cftime=True)

In [ ]:
for model in historical_msftmz_cmip6.keys():
    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/historical/{model}/"
    if not os.path.exists(outpath):
        os.makedirs(outpath)
    basin = 1
    if model not in ['mpi-esm1-2-hr','cas-esm2-0', 'mpi-esm-1-2-ham', 'e3sm-1-0', 'e3sm-1-1-eca', 'icon-esm-lr']: # only mpi models have basin = 0 for the atlantic
        basin = 0
    for rea in historical_msftmz_cmip6[model].keys():
        msftmz = historical_msftmz_cmip6[model][rea]
        if len(msftmz.time.dt.year.values)==1980:
            if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
                amoc_yr = weighted_mon_to_year_mean(msftmz.sel(basin=basin, drop=True).sel(lat=26.5, method='nearest', drop=True), "msftmz")
                max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftmz
                amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
                amoc_yr = amoc_yr.load() 
                amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
# dictionary realiz for each model
cmip6_historical_amoc_dict = {}
for model in historical_msftmz_cmip6.keys():
    cmip6_historical_amoc_dict[model] = []
    for rea in historical_msftmz_cmip6[model].keys():
        msftmz = historical_msftmz_cmip6[model][rea]
        if len(msftmz.time.dt.year.values)==1980:
            cmip6_historical_amoc_dict[model].append(rea)
    if not cmip6_historical_amoc_dict[model]: #remove key from dict if realiz list empty 
        del cmip6_historical_amoc_dict[model]

with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/historical/cmip6_historical_amoc_dict.json", 'w') as json_file:
    json.dump(cmip6_historical_amoc_dict, json_file)

### piControl

In [ ]:
picontrol_facet = freva.facet_search(experiment="picontrol", project="cmip6", variable="msftmz")

In [ ]:
picontrol_msftmz_cmip6 = {}
for model in picontrol_facet['model']:
    realiz_facet = freva.facet_search(experiment="picontrol", model=model, project="cmip6", variable="msftmz")
    picontrol_msftmz_cmip6[model] = {}
    for rea in realiz_facet['ensemble']:
        files = freva.databrowser(
            ensemble=rea,experiment="picontrol", model=model, project="cmip6", variable="msftmz")
        picontrol_msftmz_cmip6[model][rea] = xr.open_mfdataset(files, parallel=True, use_cftime=True)

In [ ]:
for model in picontrol_msftmz_cmip6.keys():
    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/picontrol/{model}/"
    if not os.path.exists(outpath):
        os.makedirs(outpath)
    basin = 1
    if model not in ['mpi-esm1-2-hr', 'mpi-esm1-2-hr', 'icon-esm-lr']: # only mpi models have basin = 0 for the atlantic
        basin = 0
    for rea in picontrol_msftmz_cmip6[model].keys():
        msftmz = picontrol_msftmz_cmip6[model][rea]
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            amoc_yr = weighted_mon_to_year_mean(msftmz.sel(basin=basin, drop=True).sel(lat=26.5, method='nearest', drop=True), "msftmz")
            max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftmz
            amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
# dictionary realiz for each model
cmip6_picontrol_amoc_dict = {}
for model in picontrol_msftmz_cmip6.keys():
    cmip6_picontrol_amoc_dict[model] = []
    for rea in picontrol_msftmz_cmip6[model].keys():
        msftmz = picontrol_msftmz_cmip6[model][rea]
        cmip6_picontrol_amoc_dict[model].append(rea)
    if not cmip6_picontrol_amoc_dict[model]: #remove key from dict if realiz list empty 
        del cmip6_picontrol_amoc_dict[model]

with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/picontrol/cmip6_picontrol_amoc_dict.json", 'w') as json_file:
    json.dump(cmip6_picontrol_amoc_dict, json_file)

## SAT

### ssp126

In [ ]:
ssp126_facet = freva.facet_search(
    experiment="ssp126", product="scenariomip", project="cmip6", 
    variable="tas", time_frequency="mon", time="2015 to 2100", time_select="strict")

In [ ]:
ssp126_tas_cmip6 = {}
for model in ssp126_facet['model']:
    if not model=='mpi-esm1-2-lr': # already processed before
        realiz_facet = freva.facet_search(experiment="ssp126", model=model, product="scenariomip", project="cmip6", variable="tas", time_frequency="mon", time="2015 to 2100", time_select="strict")
        ssp126_tas_cmip6[model] = {}
        for rea in realiz_facet['ensemble']:
            files = freva.databrowser(
                ensemble=rea,experiment="ssp126", model=model, product="scenariomip", project="cmip6", 
                variable="tas", time_frequency="mon", time="2015 to 2100", time_select="strict")
            ssp126_tas_cmip6[model][rea] = xr.open_mfdataset(files, parallel=True, use_cftime=True)

In [ ]:
for model in ssp126_tas_cmip6.keys():
    if model in cmip6_ssp126_amoc_dict.keys():
        outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/{model}/"
        for rea in ssp126_tas_cmip6[model].keys():
            if rea in cmip6_ssp126_amoc_dict[model]:
                tas = ssp126_tas_cmip6[model][rea]
                if len(tas.time.dt.year.values)==1032:
                    if not os.path.isfile(outpath+f"{model}_{rea}_tas_yr.nc"):
                        tas_yr = weighted_mon_to_year_mean(tas, "tas")
                        tas_yr = tas_yr.load() 
                        tas_yr.to_netcdf(outpath+f"{model}_{rea}_tas_yr.nc")

In [ ]:
# dictionary realiz for each model
cmip6_ssp126_tas_dict = {}
for model in ssp126_tas_cmip6.keys():
    if model in cmip6_ssp126_amoc_dict.keys():
        cmip6_ssp126_tas_dict[model] = []
        for rea in ssp126_tas_cmip6[model].keys():
            if rea in cmip6_ssp126_amoc_dict[model]:
                tas = ssp126_tas_cmip6[model][rea]
                if len(tas.time.dt.year.values)==1032:
                    cmip6_ssp126_tas_dict[model].append(rea)
        if not cmip6_ssp126_tas_dict[model]: #remove key from dict if realiz list empty 
            del cmip6_ssp126_tas_dict[model]

with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/cmip6_ssp126_tas_dict.json", 'w') as json_file:
    json.dump(cmip6_ssp126_tas_dict, json_file)

In [ ]:
# repeat for cesm2 from uploaded data
path = '/work/uo1075/m300817/teu_amoc/data/CMIP6/upload/cesm2/'
for model in ['cesm2']:
    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/{model}/"
    for rea in ['r10i1p1f1']:
        tas = xr.open_mfdataset(path+"tas*ssp126*"+rea+"*", parallel=True, use_cftime=True)
        #if len(tas.time.dt.year.values)==1032:
        if not os.path.isfile(outpath+f"{model}_{rea}_tas_yr.nc"):
            tas_yr = weighted_mon_to_year_mean(tas, "tas")
            tas_yr = tas_yr.load() 
            tas_yr.to_netcdf(outpath+f"{model}_{rea}_tas_yr.nc")

seasonal data

In [ ]:
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/cmip6_ssp126_amoc_dict.json", 'r') as json_file:
    cmip6_ssp126_amoc_dict = json.load(json_file)

In [ ]:
for model in ssp126_tas_cmip6.keys():
    if model in cmip6_ssp126_amoc_dict.keys():
        for rea in ssp126_tas_cmip6[model].keys():
            if rea in cmip6_ssp126_amoc_dict[model]:
                tas = ssp126_tas_cmip6[model][rea]
    
                if len(tas.time.dt.year.values)==1032:
                    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/{model}/tas_djf/"
                    if not os.path.exists(outpath):
                        os.makedirs(outpath)
                    tas_sea = tas.resample(time='QS-DEC').mean(dim='time')
                    tas_djf = tas_sea.sel(time=tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1))
                    tas_djf = reset_djf_time(tas_djf)
                    new_time = xr.cftime_range(start='2015', periods=tas_djf.sizes['time'], freq='YS', calendar='proleptic_gregorian')
                    tas_djf = tas_djf.assign_coords(time=new_time)
                    if not os.path.isfile(outpath+f"{model}_{rea}_tas_djf.nc"):
                        tas_djf.to_netcdf(outpath+f"{model}_{rea}_tas_djf.nc")
                        
                    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/{model}/tas_jja/"
                    if not os.path.exists(outpath):
                        os.makedirs(outpath)
                    tas_sea = tas.resample(time='QS').mean(dim='time')
                    tas_jja = tas_sea.sel(time=tas_sea.time.dt.season=='JJA')
                    new_time = xr.cftime_range(start='2015', periods=tas_jja.sizes['time'], freq='YS', calendar='proleptic_gregorian')
                    tas_jja = tas_jja.assign_coords(time=new_time)
                    if not os.path.isfile(outpath+f"{model}_{rea}_tas_jja.nc"):
                        tas_jja.to_netcdf(outpath+f"{model}_{rea}_tas_jja.nc")

In [ ]:
# repeat for cesm2 from uploaded data
path = '/work/uo1075/m300817/teu_amoc/data/CMIP6/upload/cesm2/'
rea = 'r10i1p1f1'
tas = xr.open_mfdataset(path+"tas*ssp126*"+rea+"*", parallel=True, use_cftime=True)
for model in ['cesm2']:
    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/{model}/tas_djf/"
    if not os.path.exists(outpath):
        os.makedirs(outpath)
    tas_sea = tas.resample(time='QS-DEC').mean(dim='time')
    tas_djf = tas_sea.sel(time=tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1))
    tas_djf = reset_djf_time(tas_djf)
    new_time = xr.cftime_range(start='2015', periods=tas_djf.sizes['time'], freq='YS', calendar='proleptic_gregorian')
    tas_djf = tas_djf.assign_coords(time=new_time)
    if not os.path.isfile(outpath+f"{model}_{rea}_tas_djf.nc"):
        tas_djf.to_netcdf(outpath+f"{model}_{rea}_tas_djf.nc")
        
    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/{model}/tas_jja/"
    if not os.path.exists(outpath):
        os.makedirs(outpath)
    tas_sea = tas.resample(time='QS').mean(dim='time')
    tas_jja = tas_sea.sel(time=tas_sea.time.dt.season=='JJA')
    new_time = xr.cftime_range(start='2015', periods=tas_jja.sizes['time'], freq='YS', calendar='proleptic_gregorian')
    tas_jja = tas_jja.assign_coords(time=new_time)
    if not os.path.isfile(outpath+f"{model}_{rea}_tas_jja.nc"):
        tas_jja.to_netcdf(outpath+f"{model}_{rea}_tas_jja.nc")

### ssp245

In [ ]:
ssp245_facet = freva.facet_search(
    experiment="ssp245", product="scenariomip", project="cmip6", 
    variable="tas", time_frequency="mon", time="2015 to 2100", time_select="strict")

In [ ]:
ssp245_tas_cmip6 = {}
for model in ssp245_facet['model']:
    if not model=='mpi-esm1-2-lr': # already processed before
        realiz_facet = freva.facet_search(experiment="ssp245", model=model, product="scenariomip", project="cmip6", variable="tas", time_frequency="mon", time="2015 to 2100", time_select="strict")
        ssp245_tas_cmip6[model] = {}
        for rea in realiz_facet['ensemble']:
            files = freva.databrowser(
                ensemble=rea,experiment="ssp245", model=model, product="scenariomip", project="cmip6", 
                variable="tas", time_frequency="mon", time="2015 to 2100", time_select="strict")
            ssp245_tas_cmip6[model][rea] = xr.open_mfdataset(files, parallel=True, use_cftime=True)

In [ ]:
for model in ssp245_tas_cmip6.keys():
    if model in cmip6_ssp245_amoc_dict.keys():
        outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/{model}/"
        for rea in ssp245_tas_cmip6[model].keys():
            if rea in cmip6_ssp245_amoc_dict[model]:
                tas = ssp245_tas_cmip6[model][rea]
                if len(tas.time.dt.year.values)==1032:
                    if not os.path.isfile(outpath+f"{model}_{rea}_tas_yr.nc"):
                        tas_yr = weighted_mon_to_year_mean(tas, "tas")
                        tas_yr = tas_yr.load() 
                        tas_yr.to_netcdf(outpath+f"{model}_{rea}_tas_yr.nc")

In [ ]:
# dictionary realiz for each model
cmip6_ssp245_tas_dict = {}
for model in ssp245_tas_cmip6.keys():
    if model in cmip6_ssp245_amoc_dict.keys():
        cmip6_ssp245_tas_dict[model] = []
        for rea in ssp245_tas_cmip6[model].keys():
            if rea in cmip6_ssp245_amoc_dict[model]:
                tas = ssp245_tas_cmip6[model][rea]
                if len(tas.time.dt.year.values)==1032:
                    cmip6_ssp245_tas_dict[model].append(rea)
        if not cmip6_ssp245_tas_dict[model]: #remove key from dict if realiz list empty 
            del cmip6_ssp245_tas_dict[model]

with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/cmip6_ssp245_tas_dict.json", 'w') as json_file:
    json.dump(cmip6_ssp245_tas_dict, json_file)

seasonal data

In [ ]:
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/cmip6_ssp245_amoc_dict.json", 'r') as json_file:
    cmip6_ssp245_amoc_dict = json.load(json_file)

In [ ]:
for model in ssp245_tas_cmip6.keys():
    if model in cmip6_ssp245_amoc_dict.keys():
        for rea in ssp245_tas_cmip6[model].keys():
            if rea in cmip6_ssp245_amoc_dict[model]:
                tas = ssp245_tas_cmip6[model][rea]
    
                if len(tas.time.dt.year.values)==1032:
                    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/{model}/tas_djf/"
                    if not os.path.exists(outpath):
                        os.makedirs(outpath)
                    tas_sea = tas.resample(time='QS-DEC').mean(dim='time')
                    tas_djf = tas_sea.sel(time=tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1))
                    tas_djf = reset_djf_time(tas_djf)
                    new_time = xr.cftime_range(start='2015', periods=tas_djf.sizes['time'], freq='YS', calendar='proleptic_gregorian')
                    tas_djf = tas_djf.assign_coords(time=new_time)
                    if not os.path.isfile(outpath+f"{model}_{rea}_tas_djf.nc"):
                        tas_djf.to_netcdf(outpath+f"{model}_{rea}_tas_djf.nc")
                        
                    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/{model}/tas_jja/"
                    if not os.path.exists(outpath):
                        os.makedirs(outpath)
                    tas_sea = tas.resample(time='QS').mean(dim='time')
                    tas_jja = tas_sea.sel(time=tas_sea.time.dt.season=='JJA')
                    new_time = xr.cftime_range(start='2015', periods=tas_jja.sizes['time'], freq='YS', calendar='proleptic_gregorian')
                    tas_jja = tas_jja.assign_coords(time=new_time)
                    if not os.path.isfile(outpath+f"{model}_{rea}_tas_jja.nc"):
                        tas_jja.to_netcdf(outpath+f"{model}_{rea}_tas_jja.nc")

### ssp370

In [ ]:
ssp370_facet = freva.facet_search(
    experiment="ssp370", product="scenariomip", project="cmip6", 
    variable="tas", time_frequency="mon", time="2015 to 2100", time_select="strict")

In [ ]:
ssp370_tas_cmip6 = {}
for model in ssp370_facet['model']:
    if not model=='mpi-esm1-2-lr': # already processed before
        realiz_facet = freva.facet_search(experiment="ssp370", model=model, product="scenariomip", project="cmip6", variable="tas", time_frequency="mon", time="2015 to 2100", time_select="strict")
        ssp370_tas_cmip6[model] = {}
        for rea in realiz_facet['ensemble']:
            files = freva.databrowser(
                ensemble=rea,experiment="ssp370", model=model, product="scenariomip", project="cmip6", 
                variable="tas", time_frequency="mon", time="2015 to 2100", time_select="strict")
            ssp370_tas_cmip6[model][rea] = xr.open_mfdataset(files, parallel=True, use_cftime=True)

In [ ]:
for model in ssp370_tas_cmip6.keys():
    if model in cmip6_ssp370_amoc_dict.keys():
        outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/{model}/"
        for rea in ssp370_tas_cmip6[model].keys():
            if rea in cmip6_ssp370_amoc_dict[model]:
                tas = ssp370_tas_cmip6[model][rea]
                if len(tas.time.dt.year.values)==1032:
                    if not os.path.isfile(outpath+f"{model}_{rea}_tas_yr.nc"):
                        tas_yr = weighted_mon_to_year_mean(tas, "tas")
                        tas_yr = tas_yr.load() 
                        tas_yr.to_netcdf(outpath+f"{model}_{rea}_tas_yr.nc")

In [ ]:
# dictionary realiz for each model
cmip6_ssp370_tas_dict = {}
for model in ssp370_tas_cmip6.keys():
    if model in cmip6_ssp370_amoc_dict.keys():
        cmip6_ssp370_tas_dict[model] = []
        for rea in ssp370_tas_cmip6[model].keys():
            if rea in cmip6_ssp370_amoc_dict[model]:
                tas = ssp370_tas_cmip6[model][rea]
                if len(tas.time.dt.year.values)==1032:
                    cmip6_ssp370_tas_dict[model].append(rea)
        if not cmip6_ssp370_tas_dict[model]: #remove key from dict if realiz list empty 
            del cmip6_ssp370_tas_dict[model]

with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/cmip6_ssp370_tas_dict.json", 'w') as json_file:
    json.dump(cmip6_ssp370_tas_dict, json_file)

In [ ]:
# repeat for cesm2 from uploaded data
path = '/work/uo1075/m300817/teu_amoc/data/CMIP6/upload/cesm2/'
for model in ['cesm2']:
    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/{model}/"
    for rea in ['r4i1p1f1', 'r10i1p1f1', 'r11i1p1f1']:
        tas = xr.open_mfdataset(path+"tas*ssp370*"+rea+"*", parallel=True, use_cftime=True)
        #if len(tas.time.dt.year.values)==1032:
        if not os.path.isfile(outpath+f"{model}_{rea}_tas_yr.nc"):
            tas_yr = weighted_mon_to_year_mean(tas, "tas")
            tas_yr = tas_yr.load() 
            tas_yr.to_netcdf(outpath+f"{model}_{rea}_tas_yr.nc")

seasonal data

In [ ]:
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/cmip6_ssp370_amoc_dict.json", 'r') as json_file:
    cmip6_ssp370_amoc_dict = json.load(json_file)

In [ ]:
for model in ssp370_tas_cmip6.keys():
    if model in cmip6_ssp370_amoc_dict.keys():
        for rea in ssp370_tas_cmip6[model].keys():
            if rea in cmip6_ssp370_amoc_dict[model]:
                tas = ssp370_tas_cmip6[model][rea]
    
                if len(tas.time.dt.year.values)==1032:
                    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/{model}/tas_djf/"
                    if not os.path.exists(outpath):
                        os.makedirs(outpath)
                    tas_sea = tas.resample(time='QS-DEC').mean(dim='time')
                    tas_djf = tas_sea.sel(time=tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1))
                    tas_djf = reset_djf_time(tas_djf)
                    new_time = xr.cftime_range(start='2015', periods=tas_djf.sizes['time'], freq='YS', calendar='proleptic_gregorian')
                    tas_djf = tas_djf.assign_coords(time=new_time)
                    if not os.path.isfile(outpath+f"{model}_{rea}_tas_djf.nc"):
                        tas_djf.to_netcdf(outpath+f"{model}_{rea}_tas_djf.nc")
                        
                    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/{model}/tas_jja/"
                    if not os.path.exists(outpath):
                        os.makedirs(outpath)
                    tas_sea = tas.resample(time='QS').mean(dim='time')
                    tas_jja = tas_sea.sel(time=tas_sea.time.dt.season=='JJA')
                    new_time = xr.cftime_range(start='2015', periods=tas_jja.sizes['time'], freq='YS', calendar='proleptic_gregorian')
                    tas_jja = tas_jja.assign_coords(time=new_time)
                    if not os.path.isfile(outpath+f"{model}_{rea}_tas_jja.nc"):
                        tas_jja.to_netcdf(outpath+f"{model}_{rea}_tas_jja.nc")

In [ ]:
# repeat for cesm2 from uploaded data
path = '/work/uo1075/m300817/teu_amoc/data/CMIP6/upload/cesm2/'
for model in ['cesm2']:
    for rea in ['r4i1p1f1', 'r10i1p1f1', 'r11i1p1f1']:
        tas = xr.open_mfdataset(path+"tas*ssp370*"+rea+"*", parallel=True, use_cftime=True)
        outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/{model}/tas_djf/"
        if not os.path.exists(outpath):
            os.makedirs(outpath)
        tas_sea = tas.resample(time='QS-DEC').mean(dim='time')
        tas_djf = tas_sea.sel(time=tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1))
        tas_djf = reset_djf_time(tas_djf)
        new_time = xr.cftime_range(start='2015', periods=tas_djf.sizes['time'], freq='YS', calendar='proleptic_gregorian')
        tas_djf = tas_djf.assign_coords(time=new_time)
        if not os.path.isfile(outpath+f"{model}_{rea}_tas_djf.nc"):
            tas_djf.to_netcdf(outpath+f"{model}_{rea}_tas_djf.nc")
            
        outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/{model}/tas_jja/"
        if not os.path.exists(outpath):
            os.makedirs(outpath)
        tas_sea = tas.resample(time='QS').mean(dim='time')
        tas_jja = tas_sea.sel(time=tas_sea.time.dt.season=='JJA')
        new_time = xr.cftime_range(start='2015', periods=tas_jja.sizes['time'], freq='YS', calendar='proleptic_gregorian')
        tas_jja = tas_jja.assign_coords(time=new_time)
        if not os.path.isfile(outpath+f"{model}_{rea}_tas_jja.nc"):
            tas_jja.to_netcdf(outpath+f"{model}_{rea}_tas_jja.nc")

### historical

In [ ]:
historical_facet = freva.facet_search(
    experiment="historical", product="cmip", project="cmip6", 
    variable="tas", time_frequency="mon", time="1850 to 2014", time_select="strict")

In [ ]:
historical_tas_cmip6 = {}
for model in historical_facet['model']:
    if model not in ['mpi-esm1-2-lr', 'ec-earth3', 'ipsl-cm6a-lr']: # already processed before or problematic
        realiz_facet = freva.facet_search(experiment="historical", model=model, product="cmip", project="cmip6", variable="tas", time_frequency="mon", time="1850 to 2014", time_select="strict")
        historical_tas_cmip6[model] = {}
        for rea in realiz_facet['ensemble']:
            files = freva.databrowser(
                ensemble=rea,experiment="historical", model=model, product="cmip", project="cmip6", 
                variable="tas", time_frequency="mon", time="1850 to 2014", time_select="strict")
            historical_tas_cmip6[model][rea] = xr.open_mfdataset(files, parallel=True, use_cftime=True)

In [ ]:
for model in historical_tas_cmip6.keys():
    if model in cmip6_historical_amoc_dict.keys():
        outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/historical/{model}/"
        for rea in historical_tas_cmip6[model].keys():
            if rea in cmip6_historical_amoc_dict[model]:
                tas = historical_tas_cmip6[model][rea]
                if len(tas.time.dt.year.values)==1980:
                    if not os.path.isfile(outpath+f"{model}_{rea}_tas_yr.nc"):
                        tas_yr = weighted_mon_to_year_mean(tas, "tas")
                        tas_yr = tas_yr.load() 
                        tas_yr.to_netcdf(outpath+f"{model}_{rea}_tas_yr.nc")

In [ ]:
# dictionary realiz for each model
cmip6_historical_tas_dict = {}
for model in historical_tas_cmip6.keys():
    if model in cmip6_historical_amoc_dict.keys():
        cmip6_historical_tas_dict[model] = []
        for rea in historical_tas_cmip6[model].keys():
            if rea in cmip6_historical_amoc_dict[model]:
                tas = historical_tas_cmip6[model][rea]
                if len(tas.time.dt.year.values)==1980:
                    cmip6_historical_tas_dict[model].append(rea)
        if not cmip6_historical_tas_dict[model]: #remove key from dict if realiz list empty 
            del cmip6_historical_tas_dict[model]
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/historical/cmip6_historical_tas_dict.json", 'w') as json_file:
    json.dump(cmip6_historical_tas_dict, json_file)

### piControl

In [ ]:
picontrol_facet = freva.facet_search(experiment="picontrol", project="cmip6", variable="tas", time_frequency="mon")

In [ ]:
picontrol_tas_cmip6 = {}
for model in picontrol_facet['model']:
    realiz_facet = freva.facet_search(experiment="picontrol", model=model, project="cmip6", variable="tas", time_frequency="mon")
    picontrol_tas_cmip6[model] = {}
    for rea in realiz_facet['ensemble']:
        files = freva.databrowser(
            ensemble=rea,experiment="picontrol", model=model, project="cmip6", variable="tas", time_frequency="mon")
        try:
            picontrol_tas_cmip6[model][rea] = xr.open_mfdataset(files, parallel=True, use_cftime=True)
        except:
            continue

In [ ]:
for model in picontrol_tas_cmip6.keys():
    if model in cmip6_picontrol_amoc_dict.keys():
        outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/picontrol/{model}/"
        for rea in picontrol_tas_cmip6[model].keys():
            if rea in cmip6_picontrol_amoc_dict[model]:
                tas = picontrol_tas_cmip6[model][rea]
                if not os.path.isfile(outpath+f"{model}_{rea}_tas_yr.nc"):
                    tas_yr = weighted_mon_to_year_mean(tas, "tas")
                    tas_yr = tas_yr.load() 
                    tas_yr.to_netcdf(outpath+f"{model}_{rea}_tas_yr.nc")

In [ ]:
# dictionary realiz for each model
cmip6_picontrol_tas_dict = {}
for model in picontrol_tas_cmip6.keys():
    if model in cmip6_picontrol_amoc_dict.keys():
        cmip6_picontrol_tas_dict[model] = []
        for rea in picontrol_tas_cmip6[model].keys():
            if rea in cmip6_picontrol_amoc_dict[model]:
                tas = picontrol_tas_cmip6[model][rea]
                cmip6_picontrol_tas_dict[model].append(rea)
        if not cmip6_picontrol_tas_dict[model]: #remove key from dict if realiz list empty 
            del cmip6_picontrol_tas_dict[model]

with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/picontrol/cmip6_picontrol_tas_dict.json", 'w') as json_file:
    json.dump(cmip6_picontrol_tas_dict, json_file)

seasonal data

In [ ]:
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/picontrol/cmip6_picontrol_amoc_dict.json", 'r') as json_file:
    cmip6_picontrol_amoc_dict = json.load(json_file)

In [ ]:
for model in picontrol_tas_cmip6.keys():
    if model in cmip6_picontrol_amoc_dict.keys():
        for rea in picontrol_tas_cmip6[model].keys():
            if rea in cmip6_picontrol_amoc_dict[model]:
                tas = picontrol_tas_cmip6[model][rea]
    
                outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/picontrol/{model}/tas_djf/"
                if not os.path.exists(outpath):
                    os.makedirs(outpath)
                tas_sea = tas.resample(time='QS-DEC').mean(dim='time')
                tas_djf = tas_sea.sel(time=tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1))
                tas_djf = reset_djf_time(tas_djf)
                new_time = xr.cftime_range(start='0001', periods=tas_djf.sizes['time'], freq='YS', calendar='proleptic_gregorian')
                tas_djf = tas_djf.assign_coords(time=new_time)
                if not os.path.isfile(outpath+f"{model}_{rea}_tas_djf.nc"):
                    tas_djf.to_netcdf(outpath+f"{model}_{rea}_tas_djf.nc")
                    
                outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/picontrol/{model}/tas_jja/"
                if not os.path.exists(outpath):
                    os.makedirs(outpath)
                tas_sea = tas.resample(time='QS').mean(dim='time')
                tas_jja = tas_sea.sel(time=tas_sea.time.dt.season=='JJA')
                new_time = xr.cftime_range(start='0001', periods=tas_jja.sizes['time'], freq='YS', calendar='proleptic_gregorian')
                tas_jja = tas_jja.assign_coords(time=new_time)
                if not os.path.isfile(outpath+f"{model}_{rea}_tas_jja.nc"):
                    tas_jja.to_netcdf(outpath+f"{model}_{rea}_tas_jja.nc")

In [ ]:
# repeat for EC-Earth3
for model in ['ec-earth3']:
    for rea in picontrol_tas_cmip6[model].keys():
        tas = picontrol_tas_cmip6[model][rea]

        outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/picontrol/{model}/tas_djf/"
        if not os.path.exists(outpath):
            os.makedirs(outpath)
        tas_sea = tas.resample(time='QS-DEC').mean(dim='time')
        tas_djf = tas_sea.sel(time=tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1))
        tas_djf = reset_djf_time(tas_djf)
        new_time = xr.cftime_range(start='0001', periods=tas_djf.sizes['time'], freq='YS', calendar='proleptic_gregorian')
        tas_djf = tas_djf.assign_coords(time=new_time)
        if not os.path.isfile(outpath+f"{model}_{rea}_tas_djf.nc"):
            tas_djf.to_netcdf(outpath+f"{model}_{rea}_tas_djf.nc")
            
        outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/picontrol/{model}/tas_jja/"
        if not os.path.exists(outpath):
            os.makedirs(outpath)
        tas_sea = tas.resample(time='QS').mean(dim='time')
        tas_jja = tas_sea.sel(time=tas_sea.time.dt.season=='JJA')
        new_time = xr.cftime_range(start='0001', periods=tas_jja.sizes['time'], freq='YS', calendar='proleptic_gregorian')
        tas_jja = tas_jja.assign_coords(time=new_time)
        if not os.path.isfile(outpath+f"{model}_{rea}_tas_jja.nc"):
            tas_jja.to_netcdf(outpath+f"{model}_{rea}_tas_jja.nc")

## AMOC msftyz

### ssp126

In [ ]:
ssp126_facet_msftyz = freva.facet_search(experiment="ssp126", product="scenariomip", project="cmip6", variable="msftyz", time="2015 to 2100", time_select="strict")
ssp126_facet_msftmz = freva.facet_search(experiment="ssp126", product="scenariomip", project="cmip6", variable="msftmz", time="2015 to 2100", time_select="strict")

In [ ]:
ssp126_msftyz_cmip6 = {}
for model in ssp126_facet_msftyz['model']:
    realiz_facet = freva.facet_search(experiment="ssp126", model=model, product="scenariomip", project="cmip6", variable="msftyz", time="2015 to 2100", time_select="strict")
    ssp126_msftyz_cmip6[model] = {}
    for rea in realiz_facet['ensemble']:
        files = freva.databrowser(
            ensemble=rea,experiment="ssp126", model=model, product="scenariomip", project="cmip6", 
            variable="msftyz", time="2015 to 2100", time_select="strict")
        ssp126_msftyz_cmip6[model][rea] = xr.open_mfdataset(files, parallel=True, use_cftime=True)

In [ ]:
model = 'ukesm1-0-ll'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
basin = 0
for rea in ssp126_msftyz_cmip6[model].keys():
    msftyz = ssp126_msftyz_cmip6[model][rea]
    if len(msftyz.time.dt.year.values)==1032:
        #if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            amoc_yr = weighted_mon_to_year_mean(msftyz.sel(basin=basin, drop=True).sel(rlat=26.5, method='nearest', drop=True), "msftyz")
            max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftyz
            amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
model = 'hadgem3-gc31-ll'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
basin = 0
for rea in ssp126_msftyz_cmip6[model].keys():
    msftyz = ssp126_msftyz_cmip6[model][rea]
    if len(msftyz.time.dt.year.values)==1032:
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            amoc_yr = weighted_mon_to_year_mean(msftyz.sel(basin=basin, drop=True).sel(rlat=26.5, method='nearest', drop=True), "msftyz")
            max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftyz
            amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
model = 'ipsl-cm6a-lr'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
for rea in ssp126_msftyz_cmip6[model].keys():
    msftyz = ssp126_msftyz_cmip6[model][rea]
    if len(msftyz.time.dt.year.values)==1032:
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            for y in msftyz.y.values:
                if msftyz.sel(y=y).nav_lat == 26.056036:
                    msftyz = msftyz.sel(y=y).isel(x=0)
                    break
            amoc_yr = weighted_mon_to_year_mean(msftyz.sel({"3basin": 2}, drop=True), "msftyz")
            max_olevel_idx= amoc_yr.idxmax(dim='olevel') #select depths with maximum msftmz
            amoc_yr = (amoc_yr.sel(olevel=max_olevel_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
# cnrm-cm6-1 & cnrm-esm2-1 lat values taken from ESMValCore (https://github.com/ESMValGroup/ESMValCore/compare/development_amoc_cmip6?expand=1)
lats_cnrm = [-78.31010985, -78.12948353, -77.94582993, -77.75905569, -77.56906686, -77.37576547, -77.17905289, -76.97882346,
            -76.7749755 , -76.56740085, -76.35598816, -76.14062679, -75.92120235, -75.69759868, -75.46970003, -75.23738747,
            -75.00054051, -74.74718467, -74.50211201, -74.25207444, -73.99695178, -73.73662761, -73.47098739, -73.19991796,
            -72.92330808, -72.6410544 , -72.353054  , -72.05921232, -71.75943931, -71.45365132, -71.14177225, -70.82373504,
            -70.49948169, -70.16895926, -69.83213178, -69.48896532, -69.13944282, -68.78355532, -68.42130258, -68.05269661,
            -67.67775775, -67.29650582, -66.90862274, -66.51325989, -66.11151886, -65.70331573, -65.28856659, -64.86719513,
            -64.43910217, -64.00422668, -63.56246948, -63.11375427, -62.65800095, -62.19512558, -61.72504807, -61.24769211,
            -60.76297379, -60.27082062, -59.77114868, -59.2638855 , -58.74895477, -58.22628403, -57.69579697, -57.15742874,
            -56.61111069, -56.05677032, -55.49434662, -54.92377472, -54.34499359, -53.75794983, -53.1625824 , -52.55883789,
            -51.94667435, -51.32603455, -50.69688416, -50.0591774 , -49.41287994, -48.75795746, -48.09438705, -47.42214203,
            -46.74119949, -46.051548  , -45.35317612, -44.6460762 , -43.93025208, -43.20571136, -42.47245789, -41.73051071,
            -40.97989655, -40.22064209, -39.45278549, -38.67636108, -37.89142609, -37.09803009, -36.29623795, -35.48611832,
            -34.66775131, -33.84122086, -33.00661469, -32.16403961, -31.31359863, -30.4554081 , -29.58959389, -28.7162838 ,
            -27.83562279, -26.94775391, -26.05283546, -25.15103149, -24.24251175, -23.32746124, -22.40606308, -21.47851562,
            -20.54502296, -19.605793  , -18.66210938, -17.71509743, -16.76747131, -15.82362843, -14.88963032, -13.97249222,
            -13.07900047, -12.21489906, -11.3851366 , -10.59480476, -9.84941292,  -9.1537075 ,  -8.50995922,  -7.91714287,
            -7.3714118 ,  -6.86724281,  -6.39854765,  -5.95941591,  -5.54450321,  -5.1491704 ,  -4.76948118,  -4.40214634,
            -4.04443884,  -3.69411659,  -3.3493495 ,  -3.00865722,  -2.67085862,  -2.33502841,  -2.00046086,  -1.66663659,
            -1.33319354,  -0.99990022,  -0.66662848,  -0.33332792,  0.        ,   0.33332792,   0.66662848,   0.99990022,
             1.33319354,   1.66663659,   2.00046086,   2.33502841,  2.67085862,   3.00865722,   3.3493495 ,   3.69411659,
             4.04443884,   4.40214634,   4.76948118,   5.1491704 ,  5.54450321,   5.95941591,   6.39854765,   6.86724281,
             7.3714118 ,   7.91714287,   8.50995922,   9.1537075 ,  9.84941292,  10.59480476,  11.3851366 ,  12.21489906,
            13.07900047,  13.97249222,  14.88963032,  15.82362843,  16.76747131,  17.71509743,  18.66210938,  19.605793  ,
            20.54499518,  21.4779822 ,  22.40397865,  23.32236149,  24.23251133,  25.13383794,  26.0257901 ,  26.90785984,
            27.77958485,  28.64055043,  29.49038969,  30.32878485,  31.15546609,  31.97021166,  32.77284639,  33.56324014,
            34.3413056 ,  35.1069964 ,  35.86030526,  36.60126   ,  37.32992187,  38.04638267,  38.750761  ,  39.44320076,
            40.12386792,  40.79294734,  41.45064064,  42.09716415,  42.73274625,  43.35762528,  43.97204697,  44.57626374,
            45.17053258,  45.75511325,  46.33026766,  46.89625695,  47.45334275,  48.0017845 ,  48.54183939,  49.07376113,
            49.59779916,  50.11419933,  50.62320079,  51.12503869,  51.61994169,  52.10813175,  52.58982474,  53.06522926,
            53.53454666,  53.99797104,  54.45568858,  54.90787732,  55.35470726,  55.79633995,  56.23292769,  56.66461435,
            57.09153389,  57.51381116,  57.93155986,  58.34488469,  58.75387887,  59.15862474,  59.55919177,  59.95563977,
            60.34801363,  60.73634652,  61.12065795,  61.50095275,  61.87722169,  62.24944061,  62.6175688 ,  62.98155002,
            63.3413111 ,  63.6967619 ,  64.04779419,  64.39428222,  64.73608175,  65.07303015,  65.40494546,  65.73162743,
            66.05285733,  66.36839722,  66.67799154,  66.98136628,  67.27823129,  67.5682795 ,  67.85118942,  68.12662456,
            68.39423657,  68.65366673,  68.90454521,  69.14649675,  69.37914038,  69.6020903 ,  69.81496214,  70.01737144,
            70.20893729,  70.38928469,  70.55804591,  70.71486276,  70.85938872,  70.9912902 ,  71.11024871,  71.21595922,
            71.30812349,  71.38640637,  71.45032307,  71.49904241,  71.53102143,  71.53102144]

In [ ]:
model = 'cnrm-cm6-1'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
basin = 1
for rea in ssp126_msftyz_cmip6[model].keys():
    msftyz = ssp126_msftyz_cmip6[model][rea].assign_coords(**{'j-mean': lats_cnrm}).rename({'j-mean': 'lat'})
    if len(msftyz.time.dt.year.values)==1032:
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            amoc_yr = weighted_mon_to_year_mean(msftyz.sel(basin=basin, drop=True).sel(lat=26.5, method='nearest', drop=True), "msftyz")
            max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftyz
            amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
model = 'cnrm-esm2-1'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
basin = 1
for rea in ssp126_msftyz_cmip6[model].keys():
    msftyz = ssp126_msftyz_cmip6[model][rea].assign_coords(**{'j-mean': lats_cnrm}).rename({'j-mean': 'lat'})
    if len(msftyz.time.dt.year.values)==1032:
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            amoc_yr = weighted_mon_to_year_mean(msftyz.sel(basin=basin, drop=True).sel(lat=26.5, method='nearest', drop=True), "msftyz")
            max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftyz
            amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
# dictionary realiz for each model
cmip6_ssp126_amoc_msftyz_dict = {}
for model in ssp126_msftyz_cmip6.keys():
    cmip6_ssp126_amoc_msftyz_dict[model] = []
    for rea in ssp126_msftyz_cmip6[model].keys():
        msftyz = ssp126_msftyz_cmip6[model][rea]
        if len(msftyz.time.dt.year.values)==1032:
            cmip6_ssp126_amoc_msftyz_dict[model].append(rea)
    if not cmip6_ssp126_amoc_msftyz_dict[model]: #remove key from dict if realiz list empty 
        del cmip6_ssp126_amoc_msftyz_dict[model]
del cmip6_ssp126_amoc_msftyz_dict['cnrm-cm6-1-hr']
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/cmip6_ssp126_amoc_msftyz_dict.json", 'w') as json_file:
    json.dump(cmip6_ssp126_amoc_msftyz_dict, json_file)

In [ ]:
# repeat for hadgem3-gc31-mm from uploaded data
path = '/work/uo1075/m300817/teu_amoc/data/CMIP6/upload/hadgem3-gc31-mm/'
basin = 0 #'atlantic_arctic_ocean'
for model in ['hadgem3-gc31-mm']:
    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/{model}/"
    for rea in ['r1i1p1f3']:
        msftyz = xr.open_mfdataset(path+"msftyz*ssp126*"+rea+"*", parallel=True, use_cftime=True)
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            amoc_yr = weighted_mon_to_year_mean(msftyz.isel(basin=basin, drop=True).sel(rlat=26.5, method='nearest', drop=True), "msftyz")
            max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftyz
            amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

### ssp245

In [ ]:
ssp245_facet_msftyz = freva.facet_search(experiment="ssp245", product="scenariomip", project="cmip6", variable="msftyz", time="2015 to 2100", time_select="strict")
ssp245_facet_msftmz = freva.facet_search(experiment="ssp245", product="scenariomip", project="cmip6", variable="msftmz", time="2015 to 2100", time_select="strict")

In [ ]:
ssp245_msftyz_cmip6 = {}
for model in ssp245_facet_msftyz['model']:
    realiz_facet = freva.facet_search(experiment="ssp245", model=model, product="scenariomip", project="cmip6", variable="msftyz", time="2015 to 2100", time_select="strict")
    ssp245_msftyz_cmip6[model] = {}
    for rea in realiz_facet['ensemble']:
        files = freva.databrowser(
            ensemble=rea,experiment="ssp245", model=model, product="scenariomip", project="cmip6", 
            variable="msftyz", time="2015 to 2100", time_select="strict")
        ssp245_msftyz_cmip6[model][rea] = xr.open_mfdataset(files, parallel=True, use_cftime=True)

In [ ]:
model = 'ukesm1-0-ll'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
basin = 0
for rea in ssp245_msftyz_cmip6[model].keys():
    msftyz = ssp245_msftyz_cmip6[model][rea]
    if len(msftyz.time.dt.year.values)==1032:
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            amoc_yr = weighted_mon_to_year_mean(msftyz.sel(basin=basin, drop=True).sel(rlat=26.5, method='nearest', drop=True), "msftyz")
            max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftyz
            amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
model = 'hadgem3-gc31-ll'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
basin = 0
for rea in ssp245_msftyz_cmip6[model].keys():
    msftyz = ssp245_msftyz_cmip6[model][rea]
    if len(msftyz.time.dt.year.values)==1032:
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            amoc_yr = weighted_mon_to_year_mean(msftyz.sel(basin=basin, drop=True).sel(rlat=26.5, method='nearest', drop=True), "msftyz")
            max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftyz
            amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
model = 'ipsl-cm6a-lr'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
for rea in ssp245_msftyz_cmip6[model].keys():
    msftyz = ssp245_msftyz_cmip6[model][rea]
    if len(msftyz.time.dt.year.values)==1032:
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            for y in msftyz.y.values:
                if msftyz.sel(y=y).nav_lat == 26.056036:
                    msftyz = msftyz.sel(y=y).isel(x=0)
                    break
            amoc_yr = weighted_mon_to_year_mean(msftyz.sel({"3basin": 2}, drop=True), "msftyz")
            max_olevel_idx= amoc_yr.idxmax(dim='olevel') #select depths with maximum msftmz
            amoc_yr = (amoc_yr.sel(olevel=max_olevel_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
model = 'cnrm-cm6-1'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
basin = 1
for rea in ssp245_msftyz_cmip6[model].keys():
    msftyz = ssp245_msftyz_cmip6[model][rea].assign_coords(**{'j-mean': lats_cnrm}).rename({'j-mean': 'lat'})
    if len(msftyz.time.dt.year.values)==1032:
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            amoc_yr = weighted_mon_to_year_mean(msftyz.sel(basin=basin, drop=True).sel(lat=26.5, method='nearest', drop=True), "msftyz")
            max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftyz
            amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
model = 'cnrm-esm2-1'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
basin = 1
for rea in ssp245_msftyz_cmip6[model].keys():
    msftyz = ssp245_msftyz_cmip6[model][rea].assign_coords(**{'j-mean': lats_cnrm}).rename({'j-mean': 'lat'})
    if len(msftyz.time.dt.year.values)==1032:
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            amoc_yr = weighted_mon_to_year_mean(msftyz.sel(basin=basin, drop=True).sel(lat=26.5, method='nearest', drop=True), "msftyz")
            max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftyz
            amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
# dictionary realiz for each model
cmip6_ssp245_amoc_msftyz_dict = {}
for model in ssp245_msftyz_cmip6.keys():
    cmip6_ssp245_amoc_msftyz_dict[model] = []
    for rea in ssp245_msftyz_cmip6[model].keys():
        msftyz = ssp245_msftyz_cmip6[model][rea]
        if len(msftyz.time.dt.year.values)==1032:
            cmip6_ssp245_amoc_msftyz_dict[model].append(rea)
    if not cmip6_ssp245_amoc_msftyz_dict[model]: #remove key from dict if realiz list empty 
        del cmip6_ssp245_amoc_msftyz_dict[model]
del cmip6_ssp245_amoc_msftyz_dict['cnrm-cm6-1-hr'] # Failed with cnrm-esm2-1-hr, don't know latitude values
del cmip6_ssp245_amoc_msftyz_dict['mri-esm2-0'] # mri already has msftmz

with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/cmip6_ssp245_amoc_msftyz_dict.json", 'w') as json_file:
    json.dump(cmip6_ssp245_amoc_msftyz_dict, json_file)

In [ ]:
# repeat for ec-earth3 from uploaded data
path = '/work/uo1075/m300817/teu_amoc/data/CMIP6/upload/ec-earth3/'
basin = 1 #'atlantic_arctic_ocean'
for model in ['ec-earth3']:
    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/{model}/"
    for rea in ['r2i1p1f1', 'r10i1p1f1', 'r21i1p1f1']:
        msftyz = xr.open_mfdataset(path+"msftyz*ssp245*"+rea+"*", parallel=True, use_cftime=True)
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            amoc_yr = weighted_mon_to_year_mean(msftyz.isel(basin=basin, drop=True).sel(rlat=26.5, method='nearest', drop=True), "msftyz")
            max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftyz
            amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

### ssp370

In [ ]:
ssp370_facet_msftyz = freva.facet_search(experiment="ssp370", product="scenariomip", project="cmip6", variable="msftyz", time="2015 to 2100", time_select="strict")
ssp370_facet_msftmz = freva.facet_search(experiment="ssp370", product="scenariomip", project="cmip6", variable="msftmz", time="2015 to 2100", time_select="strict")

In [ ]:
ssp370_msftyz_cmip6 = {}
for model in ssp370_facet_msftyz['model']:
    realiz_facet = freva.facet_search(experiment="ssp370", model=model, product="scenariomip", project="cmip6", variable="msftyz", time="2015 to 2100", time_select="strict")
    ssp370_msftyz_cmip6[model] = {}
    for rea in realiz_facet['ensemble']:
        files = freva.databrowser(
            ensemble=rea,experiment="ssp370", model=model, product="scenariomip", project="cmip6", 
            variable="msftyz", time="2015 to 2100", time_select="strict")
        ssp370_msftyz_cmip6[model][rea] = xr.open_mfdataset(files, parallel=True, use_cftime=True)

In [ ]:
model = 'ukesm1-0-ll'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
basin = 0
for rea in ssp370_msftyz_cmip6[model].keys():
    msftyz = ssp370_msftyz_cmip6[model][rea]
    if len(msftyz.time.dt.year.values)==1032:
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            amoc_yr = weighted_mon_to_year_mean(msftyz.sel(basin=basin, drop=True).sel(rlat=26.5, method='nearest', drop=True), "msftyz")
            max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftyz
            amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
model = 'ipsl-cm6a-lr'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
for rea in ssp370_msftyz_cmip6[model].keys():
    msftyz = ssp370_msftyz_cmip6[model][rea]
    if len(msftyz.time.dt.year.values)==1032:
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            for y in msftyz.y.values:
                if msftyz.sel(y=y).nav_lat == 26.056036:
                    msftyz = msftyz.sel(y=y).isel(x=0)
                    break
            amoc_yr = weighted_mon_to_year_mean(msftyz.sel({"3basin": 2}, drop=True), "msftyz")
            max_olevel_idx= amoc_yr.idxmax(dim='olevel') #select depths with maximum msftmz
            amoc_yr = (amoc_yr.sel(olevel=max_olevel_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
model = 'cnrm-cm6-1'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
basin = 1
for rea in ssp370_msftyz_cmip6[model].keys():
    msftyz = ssp370_msftyz_cmip6[model][rea].assign_coords(**{'j-mean': lats_cnrm}).rename({'j-mean': 'lat'})
    if len(msftyz.time.dt.year.values)==1032:
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            amoc_yr = weighted_mon_to_year_mean(msftyz.sel(basin=basin, drop=True).sel(lat=26.5, method='nearest', drop=True), "msftyz")
            max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftyz
            amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
model = 'cnrm-esm2-1'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
basin = 1
for rea in ssp370_msftyz_cmip6[model].keys():
    msftyz = ssp370_msftyz_cmip6[model][rea].assign_coords(**{'j-mean': lats_cnrm}).rename({'j-mean': 'lat'})
    if len(msftyz.time.dt.year.values)==1032:
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            amoc_yr = weighted_mon_to_year_mean(msftyz.sel(basin=basin, drop=True).sel(lat=26.5, method='nearest', drop=True), "msftyz")
            max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftyz
            amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
# dictionary realiz for each model
cmip6_ssp370_amoc_msftyz_dict = {}
for model in ssp370_msftyz_cmip6.keys():
    cmip6_ssp370_amoc_msftyz_dict[model] = []
    for rea in ssp370_msftyz_cmip6[model].keys():
        msftyz = ssp370_msftyz_cmip6[model][rea]
        if len(msftyz.time.dt.year.values)==1032:
            cmip6_ssp370_amoc_msftyz_dict[model].append(rea)
    if not cmip6_ssp370_amoc_msftyz_dict[model]: #remove key from dict if realiz list empty 
        del cmip6_ssp370_amoc_msftyz_dict[model]
del cmip6_ssp370_amoc_msftyz_dict['mri-esm2-0'] # mri already has msftmz

with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/cmip6_ssp370_amoc_msftyz_dict.json", 'w') as json_file:
    json.dump(cmip6_ssp370_amoc_msftyz_dict, json_file)

### historical 

only for those with ssps + hadgem3-gc31-mm

In [34]:
historical_facet_msftyz = freva.facet_search(experiment="historical", product="cmip", project="cmip6", variable="msftyz", time="1850 to 2014", time_select="strict")
historical_facet_msftmz = freva.facet_search(experiment="historical", product="cmip", project="cmip6", variable="msftmz", time="1850 to 2014", time_select="strict")

In [ ]:
historical_msftyz_cmip6 = {}
for model in historical_facet_msftyz['model']:
    if not model=='mpi-esm1-2-lr': # already processed before
        realiz_facet = freva.facet_search(model=model, experiment="historical", product="cmip", project="cmip6", variable="msftyz", time="1850 to 2014", time_select="strict")
        historical_msftyz_cmip6[model] = {}
        for rea in realiz_facet['ensemble']:
            files = freva.databrowser(
                ensemble=rea, model=model, experiment="historical", product="cmip", project="cmip6", 
                variable="msftyz", time="1850 to 2014", time_select="strict")
            historical_msftyz_cmip6[model][rea] = xr.open_mfdataset(files, parallel=True, use_cftime=True)

In [ ]:
model = 'ukesm1-0-ll'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/historical/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
basin = 0
for rea in historical_msftyz_cmip6[model].keys():
    msftyz = historical_msftyz_cmip6[model][rea]
    if len(msftyz.time.dt.year.values)==1980:
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            amoc_yr = weighted_mon_to_year_mean(msftyz.sel(basin=basin, drop=True).sel(rlat=26.5, method='nearest', drop=True), "msftyz")
            max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftyz
            amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
model = 'hadgem3-gc31-ll'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/historical/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
basin = 0
for rea in historical_msftyz_cmip6[model].keys():
    msftyz = historical_msftyz_cmip6[model][rea]
    if len(msftyz.time.dt.year.values)==1980:
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            amoc_yr = weighted_mon_to_year_mean(msftyz.sel(basin=basin, drop=True).sel(rlat=26.5, method='nearest', drop=True), "msftyz")
            max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftyz
            amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
model = 'hadgem3-gc31-mm'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/historical/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
basin = 0
for rea in historical_msftyz_cmip6[model].keys():
    msftyz = historical_msftyz_cmip6[model][rea]
    if len(msftyz.time.dt.year.values)==1980:
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            amoc_yr = weighted_mon_to_year_mean(msftyz.sel(basin=basin, drop=True).sel(rlat=26.5, method='nearest', drop=True), "msftyz")
            max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftyz
            amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
model = 'ipsl-cm6a-lr'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/historical/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
for rea in historical_msftyz_cmip6[model].keys():
    msftyz = historical_msftyz_cmip6[model][rea]
    if len(msftyz.time.dt.year.values)==1980:
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            for y in msftyz.y.values:
                if msftyz.sel(y=y).nav_lat == 26.056036:
                    msftyz = msftyz.sel(y=y).isel(x=0)
                    break
            amoc_yr = weighted_mon_to_year_mean(msftyz.sel({"3basin": 2}, drop=True), "msftyz")
            max_olevel_idx= amoc_yr.idxmax(dim='olevel') #select depths with maximum msftmz
            amoc_yr = (amoc_yr.sel(olevel=max_olevel_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
model = 'cnrm-cm6-1'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/historical/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
basin = 1
for rea in historical_msftyz_cmip6[model].keys():
    msftyz = historical_msftyz_cmip6[model][rea].assign_coords(**{'j-mean': lats_cnrm}).rename({'j-mean': 'lat'})
    if len(msftyz.time.dt.year.values)==1980:
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            amoc_yr = weighted_mon_to_year_mean(msftyz.sel(basin=basin, drop=True).sel(lat=26.5, method='nearest', drop=True), "msftyz")
            max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftyz
            amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
model = 'cnrm-esm2-1'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/historical/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
basin = 1
for rea in historical_msftyz_cmip6[model].keys():
    msftyz = historical_msftyz_cmip6[model][rea].assign_coords(**{'j-mean': lats_cnrm}).rename({'j-mean': 'lat'})
    if len(msftyz.time.dt.year.values)==1980:
        if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
            amoc_yr = weighted_mon_to_year_mean(msftyz.sel(basin=basin, drop=True).sel(lat=26.5, method='nearest', drop=True), "msftyz")
            max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftyz
            amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
            amoc_yr = amoc_yr.load() 
            amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
# dictionary realiz for each model
cmip6_historical_amoc_msftyz_dict = {}
for model in historical_msftyz_cmip6.keys():
    cmip6_historical_amoc_msftyz_dict[model] = []
    for rea in historical_msftyz_cmip6[model].keys():
        msftyz = historical_msftyz_cmip6[model][rea]
        if len(msftyz.time.dt.year.values)==1980:
            cmip6_historical_amoc_msftyz_dict[model].append(rea)
    if not cmip6_historical_amoc_msftyz_dict[model]: #remove key from dict if realiz list empty 
        del cmip6_historical_amoc_msftyz_dict[model]
del cmip6_historical_amoc_msftyz_dict['access-cm2'] # not in ssps
del cmip6_historical_amoc_msftyz_dict['access-esm1-5'] # not in ssps
del cmip6_historical_amoc_msftyz_dict['ciesm'] # not in ssps
del cmip6_historical_amoc_msftyz_dict['cnrm-cm6-1-hr'] # not in ssps
del cmip6_historical_amoc_msftyz_dict['cmcc-cm2-hr4'] # not in ssps
del cmip6_historical_amoc_msftyz_dict['mri-esm2-0'] # mri already has msftmz

with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/historical/cmip6_historical_amoc_msftyz_dict.json", 'w') as json_file:
    json.dump(cmip6_historical_amoc_msftyz_dict, json_file)

### picontrol

In [ ]:
picontrol_facet_msftyz = freva.facet_search(experiment="picontrol", project="cmip6", variable="msftyz")
picontrol_facet_msftmz = freva.facet_search(experiment="picontrol", project="cmip6", variable="msftmz")

In [ ]:
picontrol_msftyz_cmip6 = {}
for model in picontrol_facet_msftyz['model']:
    realiz_facet = freva.facet_search(experiment="picontrol", model=model, project="cmip6", variable="msftyz")
    picontrol_msftyz_cmip6[model] = {}
    for rea in realiz_facet['ensemble']:
        files = freva.databrowser(
            ensemble=rea,experiment="picontrol", model=model, project="cmip6", variable="msftyz")
        picontrol_msftyz_cmip6[model][rea] = xr.open_mfdataset(files, parallel=True, use_cftime=True)

only process models for nahosmip

In [ ]:
model = 'hadgem3-gc31-mm'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/picontrol/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
basin = 0
for rea in picontrol_msftyz_cmip6[model].keys():
    msftyz = picontrol_msftyz_cmip6[model][rea]
    if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
        amoc_yr = weighted_mon_to_year_mean(msftyz.sel(basin=basin, drop=True).sel(rlat=26.5, method='nearest', drop=True), "msftyz")
        max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftyz
        amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
        amoc_yr = amoc_yr.load() 
        amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
model = 'ipsl-cm6a-lr'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/picontrol/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
for rea in picontrol_msftyz_cmip6[model].keys():
    msftyz = picontrol_msftyz_cmip6[model][rea]
    if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
        for y in msftyz.y.values:
            if msftyz.sel(y=y).nav_lat == 26.056036:
                msftyz = msftyz.sel(y=y).isel(x=0)
                break
        amoc_yr = weighted_mon_to_year_mean(msftyz.sel({"3basin": 2}, drop=True), "msftyz")
        max_olevel_idx= amoc_yr.idxmax(dim='olevel') #select depths with maximum msftmz

        # print("Available olevel coordinates:", amoc_yr.olevel.values)
        # print("Max olevel indices:", max_olevel_idx.values if hasattr(max_olevel_idx, 'values') else max_olevel_idx)
        # print("Data type of max_olevel_idx:", type(max_olevel_idx))
        
        try:
            amoc_yr = (amoc_yr.sel(olevel=max_olevel_idx, drop=True))/(1025.0 * 10**6)
        except KeyError:
            # If exact match fails, use nearest neighbor
            amoc_yr = (amoc_yr.sel(olevel=max_olevel_idx, method='nearest', drop=True))/(1025.0 * 10**6)
        
        amoc_yr = amoc_yr.load() 
        amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

Now for data downloaded from CEDA

In [ ]:
model = 'hadgem3-gc31-ll'
rea = 'r1i1p1f1'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/picontrol/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
basin = 0
msftyz = xr.open_mfdataset(outpath+"*.nc", parallel=True, use_cftime=True)
if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
    amoc_yr = weighted_mon_to_year_mean(msftyz.sel(basin=basin, drop=True).sel(rlat=26.5, method='nearest', drop=True), "msftyz")
    max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftyz
    amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
    amoc_yr = amoc_yr.load() 
    amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
model = 'ec-earth3'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/picontrol/{model}/"
if not os.path.exists(outpath):
    os.makedirs(outpath)
basin = 1
for rea in ['r1i1p1f1']:
    msftyz = xr.open_mfdataset(outpath+"/msftyz/msftyz_Omon*.nc", parallel=True, use_cftime=True)
    if not os.path.isfile(outpath+f"{model}_{rea}_amoc26_yr.nc"):
        amoc_yr = weighted_mon_to_year_mean(msftyz.sel(basin=basin, drop=True).sel(rlat=26.5, method='nearest', drop=True), "msftyz")
        max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftyz
        amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
        amoc_yr = amoc_yr.load() 
        amoc_yr.to_netcdf(outpath+f"{model}_{rea}_amoc26_yr.nc")

In [ ]:
# dictionary realiz for each model
cmip6_picontrol_amoc_msftyz_dict = {}
for model in ['hadgem3-gc31-ll', 'hadgem3-gc31-mm', 'ipsl-cm6a-lr', 'ec-earth3-lr']: #picontrol_msftyz_cmip6.keys():
    cmip6_picontrol_amoc_msftyz_dict[model] = []
    if model == 'hadgem3-gc31-ll': 
        cmip6_picontrol_amoc_msftyz_dict[model].append('r1i1p1f1')
    else:    
        for rea in picontrol_msftyz_cmip6[model].keys():
            cmip6_picontrol_amoc_msftyz_dict[model].append(rea)
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/picontrol/cmip6_picontrol_amoc_msftyz_dict.json", 'w') as json_file:
    json.dump(cmip6_picontrol_amoc_msftyz_dict, json_file)

## SAT (msftyz models)

### ssp126

In [ ]:
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/cmip6_ssp126_amoc_msftyz_dict.json", 'r') as json_file:
    cmip6_ssp126_amoc_msftyz_dict = json.load(json_file)

In [ ]:
ssp126_facet = freva.facet_search(experiment="ssp126", product="scenariomip", project="cmip6", variable="tas", time_frequency="mon", time="2015 to 2100", time_select="strict")

In [ ]:
ssp126_tas_cmip6 = {}
for model in ssp126_facet['model']:
    if not model=='mpi-esm1-2-lr': # already processed before
        realiz_facet = freva.facet_search(experiment="ssp126", model=model, product="scenariomip", project="cmip6", variable="tas", time_frequency="mon", time="2015 to 2100", time_select="strict")
        ssp126_tas_cmip6[model] = {}
        for rea in realiz_facet['ensemble']:
            files = freva.databrowser(
                ensemble=rea,experiment="ssp126", model=model, product="scenariomip", project="cmip6", 
                variable="tas", time_frequency="mon", time="2015 to 2100", time_select="strict")
            ssp126_tas_cmip6[model][rea] = xr.open_mfdataset(files, parallel=True, use_cftime=True)

In [ ]:
for model in ssp126_tas_cmip6.keys():
    if model in cmip6_ssp126_amoc_msftyz_dict.keys():
        outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/{model}/"
        for rea in ssp126_tas_cmip6[model].keys():
            if rea in cmip6_ssp126_amoc_msftyz_dict[model]:
                tas = ssp126_tas_cmip6[model][rea]
                if len(tas.time.dt.year.values)==1032:
                    if not os.path.isfile(outpath+f"{model}_{rea}_tas_yr.nc"):
                        tas_yr = weighted_mon_to_year_mean(tas, "tas")
                        tas_yr = tas_yr.load() 
                        tas_yr.to_netcdf(outpath+f"{model}_{rea}_tas_yr.nc")

In [ ]:
# dictionary realiz for each model
cmip6_ssp126_tas_msftyz_dict = {}
for model in ssp126_tas_cmip6.keys():
    if model in cmip6_ssp126_amoc_msftyz_dict.keys():
        cmip6_ssp126_tas_msftyz_dict[model] = []
        for rea in ssp126_tas_cmip6[model].keys():
            if rea in cmip6_ssp126_amoc_msftyz_dict[model]:
                tas = ssp126_tas_cmip6[model][rea]
                if len(tas.time.dt.year.values)==1032:
                    cmip6_ssp126_tas_msftyz_dict[model].append(rea)
        if not cmip6_ssp126_tas_msftyz_dict[model]: #remove key from dict if realiz list empty 
            del cmip6_ssp126_tas_msftyz_dict[model]

with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/cmip6_ssp126_tas_msftyz_dict.json", 'w') as json_file:
    json.dump(cmip6_ssp126_tas_msftyz_dict, json_file)

In [ ]:
# repeat for hadgem3-gc31-mm from uploaded data
path = '/work/uo1075/m300817/teu_amoc/data/CMIP6/upload/hadgem3-gc31-mm/'
for model in ['hadgem3-gc31-mm']:
    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/{model}/"
    for rea in ['r1i1p1f3']:
        tas = xr.open_mfdataset(path+"tas*ssp126*"+rea+"*", parallel=True, use_cftime=True)
        if not os.path.isfile(outpath+f"{model}_{rea}_tas_yr.nc"):
            tas_yr = weighted_mon_to_year_mean(tas, "tas")
            tas_yr = tas_yr.load() 
            tas_yr.to_netcdf(outpath+f"{model}_{rea}_tas_yr.nc")

seasonal data

In [ ]:
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/cmip6_ssp126_amoc_msftyz_dict.json", 'r') as json_file:
    cmip6_ssp126_amoc_msftyz_dict = json.load(json_file)

In [ ]:
for model in ssp126_tas_cmip6.keys():
    if model in cmip6_ssp126_amoc_msftyz_dict.keys():
        for rea in ssp126_tas_cmip6[model].keys():
            if rea in cmip6_ssp126_amoc_msftyz_dict[model]:
                tas = ssp126_tas_cmip6[model][rea]
    
                if len(tas.time.dt.year.values)==1032:
                    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/{model}/tas_djf/"
                    if not os.path.exists(outpath):
                        os.makedirs(outpath)
                    tas_sea = tas.resample(time='QS-DEC').mean(dim='time')
                    tas_djf = tas_sea.sel(time=tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1))
                    tas_djf = reset_djf_time(tas_djf)
                    new_time = xr.cftime_range(start='2015', periods=tas_djf.sizes['time'], freq='YS', calendar='proleptic_gregorian')
                    tas_djf = tas_djf.assign_coords(time=new_time)
                    if not os.path.isfile(outpath+f"{model}_{rea}_tas_djf.nc"):
                        tas_djf.to_netcdf(outpath+f"{model}_{rea}_tas_djf.nc")
                        
                    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/{model}/tas_jja/"
                    if not os.path.exists(outpath):
                        os.makedirs(outpath)
                    tas_sea = tas.resample(time='QS').mean(dim='time')
                    tas_jja = tas_sea.sel(time=tas_sea.time.dt.season=='JJA')
                    new_time = xr.cftime_range(start='2015', periods=tas_jja.sizes['time'], freq='YS', calendar='proleptic_gregorian')
                    tas_jja = tas_jja.assign_coords(time=new_time)
                    if not os.path.isfile(outpath+f"{model}_{rea}_tas_jja.nc"):
                        tas_jja.to_netcdf(outpath+f"{model}_{rea}_tas_jja.nc")

In [ ]:
# repeat for hadgem3-gc31-mm from uploaded data
path = '/work/uo1075/m300817/teu_amoc/data/CMIP6/upload/hadgem3-gc31-mm/'
rea = 'r1i1p1f3'
tas = xr.open_mfdataset(path+"tas*ssp126*"+rea+"*", parallel=True, use_cftime=True)
for model in ['hadgem3-gc31-mm']:
    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/{model}/tas_djf/"
    if not os.path.exists(outpath):
        os.makedirs(outpath)
    tas_sea = tas.resample(time='QS-DEC').mean(dim='time')
    tas_djf = tas_sea.sel(time=tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1))
    tas_djf = reset_djf_time(tas_djf)
    new_time = xr.cftime_range(start='2015', periods=tas_djf.sizes['time'], freq='YS', calendar='proleptic_gregorian')
    tas_djf = tas_djf.assign_coords(time=new_time)
    if not os.path.isfile(outpath+f"{model}_{rea}_tas_djf.nc"):
        tas_djf.to_netcdf(outpath+f"{model}_{rea}_tas_djf.nc")
        
    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/{model}/tas_jja/"
    if not os.path.exists(outpath):
        os.makedirs(outpath)
    tas_sea = tas.resample(time='QS').mean(dim='time')
    tas_jja = tas_sea.sel(time=tas_sea.time.dt.season=='JJA')
    new_time = xr.cftime_range(start='2015', periods=tas_jja.sizes['time'], freq='YS', calendar='proleptic_gregorian')
    tas_jja = tas_jja.assign_coords(time=new_time)
    if not os.path.isfile(outpath+f"{model}_{rea}_tas_jja.nc"):
        tas_jja.to_netcdf(outpath+f"{model}_{rea}_tas_jja.nc")

### ssp245

In [ ]:
ssp245_facet = freva.facet_search(experiment="ssp245", product="scenariomip", project="cmip6", variable="tas", time_frequency="mon", time="2015 to 2100", time_select="strict")

In [ ]:
ssp245_tas_cmip6 = {}
for model in ssp245_facet['model']:
    if not model=='mpi-esm1-2-lr': # already processed before
        realiz_facet = freva.facet_search(experiment="ssp245", model=model, product="scenariomip", project="cmip6", variable="tas", time_frequency="mon", time="2015 to 2100", time_select="strict")
        ssp245_tas_cmip6[model] = {}
        for rea in realiz_facet['ensemble']:
            files = freva.databrowser(
                ensemble=rea,experiment="ssp245", model=model, product="scenariomip", project="cmip6", 
                variable="tas", time_frequency="mon", time="2015 to 2100", time_select="strict")
            ssp245_tas_cmip6[model][rea] = xr.open_mfdataset(files, parallel=True, use_cftime=True)

In [ ]:
for model in ssp245_tas_cmip6.keys():
    if model in cmip6_ssp245_amoc_msftyz_dict.keys():
        outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/{model}/"
        for rea in ssp245_tas_cmip6[model].keys():
            if rea in cmip6_ssp245_amoc_msftyz_dict[model]:
                tas = ssp245_tas_cmip6[model][rea]
                if len(tas.time.dt.year.values)==1032:
                    if not os.path.isfile(outpath+f"{model}_{rea}_tas_yr.nc"):
                        tas_yr = weighted_mon_to_year_mean(tas, "tas")
                        tas_yr = tas_yr.load() 
                        tas_yr.to_netcdf(outpath+f"{model}_{rea}_tas_yr.nc")

In [ ]:
# dictionary realiz for each model
cmip6_ssp245_tas_msftyz_dict = {}
for model in ssp245_tas_cmip6.keys():
    if model in cmip6_ssp245_amoc_msftyz_dict.keys():
        cmip6_ssp245_tas_msftyz_dict[model] = []
        for rea in ssp245_tas_cmip6[model].keys():
            if rea in cmip6_ssp245_amoc_msftyz_dict[model]:
                tas = ssp245_tas_cmip6[model][rea]
                if len(tas.time.dt.year.values)==1032:
                    cmip6_ssp245_tas_msftyz_dict[model].append(rea)
        if not cmip6_ssp245_tas_msftyz_dict[model]: #remove key from dict if realiz list empty 
            del cmip6_ssp245_tas_msftyz_dict[model]

with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/cmip6_ssp245_tas_msftyz_dict.json", 'w') as json_file:
    json.dump(cmip6_ssp245_tas_msftyz_dict, json_file)

In [ ]:
# repeat for ec-earth3 from uploaded data
path = '/work/uo1075/m300817/teu_amoc/data/CMIP6/upload/ec-earth3/'
for model in ['ec-earth3']:
    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/{model}/"
    for rea in ['r2i1p1f1', 'r10i1p1f1', 'r21i1p1f1']:
        tas = xr.open_mfdataset(path+"tas*ssp245*"+rea+"*", parallel=True, use_cftime=True)
        if not os.path.isfile(outpath+f"{model}_{rea}_tas_yr.nc"):
            tas_yr = weighted_mon_to_year_mean(tas, "tas")
            tas_yr = tas_yr.load() 
            tas_yr.to_netcdf(outpath+f"{model}_{rea}_tas_yr.nc")

seasonal data

In [ ]:
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/cmip6_ssp245_amoc_msftyz_dict.json", 'r') as json_file:
    cmip6_ssp245_amoc_msftyz_dict = json.load(json_file)

In [ ]:
for model in ssp245_tas_cmip6.keys():
    if model in cmip6_ssp245_amoc_msftyz_dict.keys():
        for rea in ssp245_tas_cmip6[model].keys():
            if rea in cmip6_ssp245_amoc_msftyz_dict[model]:
                tas = ssp245_tas_cmip6[model][rea]
    
                if len(tas.time.dt.year.values)==1032:
                    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/{model}/tas_djf/"
                    if not os.path.exists(outpath):
                        os.makedirs(outpath)
                    tas_sea = tas.resample(time='QS-DEC').mean(dim='time')
                    tas_djf = tas_sea.sel(time=tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1))
                    tas_djf = reset_djf_time(tas_djf)
                    new_time = xr.cftime_range(start='2015', periods=tas_djf.sizes['time'], freq='YS', calendar='proleptic_gregorian')
                    tas_djf = tas_djf.assign_coords(time=new_time)
                    if not os.path.isfile(outpath+f"{model}_{rea}_tas_djf.nc"):
                        tas_djf.to_netcdf(outpath+f"{model}_{rea}_tas_djf.nc")
                        
                    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/{model}/tas_jja/"
                    if not os.path.exists(outpath):
                        os.makedirs(outpath)
                    tas_sea = tas.resample(time='QS').mean(dim='time')
                    tas_jja = tas_sea.sel(time=tas_sea.time.dt.season=='JJA')
                    new_time = xr.cftime_range(start='2015', periods=tas_jja.sizes['time'], freq='YS', calendar='proleptic_gregorian')
                    tas_jja = tas_jja.assign_coords(time=new_time)
                    if not os.path.isfile(outpath+f"{model}_{rea}_tas_jja.nc"):
                        tas_jja.to_netcdf(outpath+f"{model}_{rea}_tas_jja.nc")

In [ ]:
# repeat for ec-earth3 from uploaded data
path = '/work/uo1075/m300817/teu_amoc/data/CMIP6/upload/ec-earth3/'

for model in ['ec-earth3']:
    for rea in ['r2i1p1f1', 'r10i1p1f1', 'r21i1p1f1']:
        tas = xr.open_mfdataset(path+"tas*ssp245*"+rea+"*", parallel=True, use_cftime=True)

        outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/{model}/tas_djf/"
        if not os.path.exists(outpath):
            os.makedirs(outpath)
        tas_sea = tas.resample(time='QS-DEC').mean(dim='time')
        tas_djf = tas_sea.sel(time=tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1))
        tas_djf = reset_djf_time(tas_djf)
        new_time = xr.cftime_range(start='2015', periods=tas_djf.sizes['time'], freq='YS', calendar='proleptic_gregorian')
        tas_djf = tas_djf.assign_coords(time=new_time)
        if not os.path.isfile(outpath+f"{model}_{rea}_tas_djf.nc"):
            tas_djf.to_netcdf(outpath+f"{model}_{rea}_tas_djf.nc")
            
        outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/{model}/tas_jja/"
        if not os.path.exists(outpath):
            os.makedirs(outpath)
        tas_sea = tas.resample(time='QS').mean(dim='time')
        tas_jja = tas_sea.sel(time=tas_sea.time.dt.season=='JJA')
        new_time = xr.cftime_range(start='2015', periods=tas_jja.sizes['time'], freq='YS', calendar='proleptic_gregorian')
        tas_jja = tas_jja.assign_coords(time=new_time)
        if not os.path.isfile(outpath+f"{model}_{rea}_tas_jja.nc"):
            tas_jja.to_netcdf(outpath+f"{model}_{rea}_tas_jja.nc")

### ssp370

In [ ]:
ssp370_facet = freva.facet_search(experiment="ssp370", product="scenariomip", project="cmip6", variable="tas", time_frequency="mon", time="2015 to 2100", time_select="strict")

In [ ]:
ssp370_tas_cmip6 = {}
for model in ssp370_facet['model']:
    if not model=='mpi-esm1-2-lr': # already processed before
        realiz_facet = freva.facet_search(experiment="ssp370", model=model, product="scenariomip", project="cmip6", variable="tas", time_frequency="mon", time="2015 to 2100", time_select="strict")
        ssp370_tas_cmip6[model] = {}
        for rea in realiz_facet['ensemble']:
            files = freva.databrowser(
                ensemble=rea,experiment="ssp370", model=model, product="scenariomip", project="cmip6", 
                variable="tas", time_frequency="mon", time="2015 to 2100", time_select="strict")
            ssp370_tas_cmip6[model][rea] = xr.open_mfdataset(files, parallel=True, use_cftime=True)

In [ ]:
for model in ssp370_tas_cmip6.keys():
    if model in cmip6_ssp370_amoc_msftyz_dict.keys():
        outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/{model}/"
        for rea in ssp370_tas_cmip6[model].keys():
            if rea in cmip6_ssp370_amoc_msftyz_dict[model]:
                tas = ssp370_tas_cmip6[model][rea]
                if len(tas.time.dt.year.values)==1032:
                    if not os.path.isfile(outpath+f"{model}_{rea}_tas_yr.nc"):
                        tas_yr = weighted_mon_to_year_mean(tas, "tas")
                        tas_yr = tas_yr.load() 
                        tas_yr.to_netcdf(outpath+f"{model}_{rea}_tas_yr.nc")

In [ ]:
# dictionary realiz for each model
cmip6_ssp370_tas_msftyz_dict = {}
for model in ssp370_tas_cmip6.keys():
    if model in cmip6_ssp370_amoc_msftyz_dict.keys():
        cmip6_ssp370_tas_msftyz_dict[model] = []
        for rea in ssp370_tas_cmip6[model].keys():
            if rea in cmip6_ssp370_amoc_msftyz_dict[model]:
                tas = ssp370_tas_cmip6[model][rea]
                if len(tas.time.dt.year.values)==1032:
                    cmip6_ssp370_tas_msftyz_dict[model].append(rea)
        if not cmip6_ssp370_tas_msftyz_dict[model]: #remove key from dict if realiz list empty 
            del cmip6_ssp370_tas_msftyz_dict[model]

with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/cmip6_ssp370_tas_msftyz_dict.json", 'w') as json_file:
    json.dump(cmip6_ssp370_tas_msftyz_dict, json_file)

seasonal data

In [ ]:
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/cmip6_ssp370_amoc_msftyz_dict.json", 'r') as json_file:
    cmip6_ssp370_amoc_msftyz_dict = json.load(json_file)

In [ ]:
for model in ssp370_tas_cmip6.keys():
    if model in cmip6_ssp370_amoc_msftyz_dict.keys():
        for rea in ssp370_tas_cmip6[model].keys():
            if rea in cmip6_ssp370_amoc_msftyz_dict[model]:
                tas = ssp370_tas_cmip6[model][rea]
    
                if len(tas.time.dt.year.values)==1032:
                    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/{model}/tas_djf/"
                    if not os.path.exists(outpath):
                        os.makedirs(outpath)
                    tas_sea = tas.resample(time='QS-DEC').mean(dim='time')
                    tas_djf = tas_sea.sel(time=tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1))
                    tas_djf = reset_djf_time(tas_djf)
                    new_time = xr.cftime_range(start='2015', periods=tas_djf.sizes['time'], freq='YS', calendar='proleptic_gregorian')
                    tas_djf = tas_djf.assign_coords(time=new_time)
                    if not os.path.isfile(outpath+f"{model}_{rea}_tas_djf.nc"):
                        tas_djf.to_netcdf(outpath+f"{model}_{rea}_tas_djf.nc")
                        
                    outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/{model}/tas_jja/"
                    if not os.path.exists(outpath):
                        os.makedirs(outpath)
                    tas_sea = tas.resample(time='QS').mean(dim='time')
                    tas_jja = tas_sea.sel(time=tas_sea.time.dt.season=='JJA')
                    new_time = xr.cftime_range(start='2015', periods=tas_jja.sizes['time'], freq='YS', calendar='proleptic_gregorian')
                    tas_jja = tas_jja.assign_coords(time=new_time)
                    if not os.path.isfile(outpath+f"{model}_{rea}_tas_jja.nc"):
                        tas_jja.to_netcdf(outpath+f"{model}_{rea}_tas_jja.nc")

### historical 

In [ ]:
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/historical/cmip6_historical_amoc_msftyz_dict.json", 'r') as json_file:
    cmip6_historical_amoc_msftyz_dict = json.load(json_file)

In [ ]:
historical_facet = freva.facet_search(
    experiment="historical", product="cmip", project="cmip6", 
    variable="tas", time_frequency="mon", time="1850 to 2014", time_select="strict")

In [ ]:
historical_tas_cmip6 = {}
for model in cmip6_historical_amoc_msftyz_dict.keys():
    if model not in ['mpi-esm1-2-lr', 'ipsl-cm6a-lr']: # ipsl error when loading, do separately
        realiz_facet = freva.facet_search(experiment="historical", model=model, product="cmip", project="cmip6", variable="tas", time_frequency="mon", time="1850 to 2014", time_select="strict")
        historical_tas_cmip6[model] = {}
        for rea in realiz_facet['ensemble']:
            files = freva.databrowser(
                ensemble=rea,experiment="historical", model=model, product="cmip", project="cmip6", 
                variable="tas", time_frequency="mon", time="1850 to 2014", time_select="strict")
            historical_tas_cmip6[model][rea] = xr.open_mfdataset(files, parallel=True, use_cftime=True)

In [ ]:
for model in historical_tas_cmip6.keys():
    if model in cmip6_historical_amoc_msftyz_dict.keys():
        outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/historical/{model}/"
        for rea in historical_tas_cmip6[model].keys():
            if rea in cmip6_historical_amoc_msftyz_dict[model]:
                tas = historical_tas_cmip6[model][rea]
                if len(tas.time.dt.year.values)==1980:
                    if not os.path.isfile(outpath+f"{model}_{rea}_tas_yr.nc"):
                        tas_yr = weighted_mon_to_year_mean(tas, "tas")
                        tas_yr = tas_yr.load() 
                        tas_yr.to_netcdf(outpath+f"{model}_{rea}_tas_yr.nc")

In [ ]:
model = 'ipsl-cm6a-lr'
historical_tas_ipsl = {}
for rea in ['r1i1p1f1', 'r25i1p1f1']:
    if rea == 'r1i1p1f1': # problem with r1 and freva since several uncompatible files exist
        files = '/work/ik1017/CMIP6/data/CMIP6/CMIP/IPSL/IPSL-CM6A-LR/historical/r1i1p1f1/Amon/tas/gr/v20180803/tas_Amon_IPSL-CM6A-LR_historical_r1i1p1f1_gr_185001-201412.nc'
    else:    
        files = freva.databrowser(
            ensemble=rea, experiment="historical", model=model, product="cmip", project="cmip6", 
            variable="tas", time_frequency="mon", time="1850 to 2014", time_select="strict")
    historical_tas_ipsl[rea] = xr.open_mfdataset(files, parallel=True, use_cftime=True)

In [ ]:
model = 'ipsl-cm6a-lr'
outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/historical/{model}/"
for rea in historical_tas_ipsl.keys():
    if rea in cmip6_historical_amoc_msftyz_dict[model]:
        tas = historical_tas_ipsl[rea]
        if len(tas.time.dt.year.values)==1980:
            if not os.path.isfile(outpath+f"{model}_{rea}_tas_yr.nc"):
                tas_yr = weighted_mon_to_year_mean(tas, "tas")
                tas_yr = tas_yr.load() 
                tas_yr.to_netcdf(outpath+f"{model}_{rea}_tas_yr.nc")

In [ ]:
# dictionary realiz for each model
cmip6_historical_tas_msftyz_dict = {}
for model in historical_tas_cmip6.keys():
    if model in cmip6_historical_amoc_msftyz_dict.keys():
        cmip6_historical_tas_msftyz_dict[model] = []
        for rea in historical_tas_cmip6[model].keys():
            if rea in cmip6_historical_amoc_msftyz_dict[model]:
                tas = historical_tas_cmip6[model][rea]
                if len(tas.time.dt.year.values)==1980:
                    cmip6_historical_tas_msftyz_dict[model].append(rea)
        if not cmip6_historical_tas_msftyz_dict[model]: #remove key from dict if realiz list empty 
            del cmip6_historical_tas_msftyz_dict[model]
model = 'ipsl-cm6a-lr'
cmip6_historical_tas_msftyz_dict[model] = []
for rea in historical_tas_ipsl.keys():
    if rea in cmip6_historical_amoc_msftyz_dict[model]:
        tas = historical_tas_ipsl[rea]
        if len(tas.time.dt.year.values)==1980:
            cmip6_historical_tas_msftyz_dict[model].append(rea)        
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/historical/cmip6_historical_tas_msftyz_dict.json", 'w') as json_file:
    json.dump(cmip6_historical_tas_msftyz_dict, json_file)

### piControl

In [ ]:
picontrol_facet = freva.facet_search(experiment="picontrol", project="cmip6", variable="tas", time_frequency="mon")

In [ ]:
picontrol_tas_cmip6 = {}
for model in picontrol_facet['model']:
    realiz_facet = freva.facet_search(experiment="picontrol", model=model, project="cmip6", variable="tas", time_frequency="mon")
    picontrol_tas_cmip6[model] = {}
    for rea in realiz_facet['ensemble']:
        files = freva.databrowser(
            ensemble=rea,experiment="picontrol", model=model, project="cmip6", variable="tas", time_frequency="mon")
        try:
            picontrol_tas_cmip6[model][rea] = xr.open_mfdataset(files, parallel=True, use_cftime=True)
        except:
            continue

In [ ]:
for model in picontrol_tas_cmip6.keys():
    if model in cmip6_picontrol_amoc_msftyz_dict.keys():
        outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/picontrol/{model}/"
        for rea in picontrol_tas_cmip6[model].keys():
            if rea in cmip6_picontrol_amoc_msftyz_dict[model]:
                tas = picontrol_tas_cmip6[model][rea]
                if not os.path.isfile(outpath+f"{model}_{rea}_tas_yr.nc"):
                    tas_yr = weighted_mon_to_year_mean(tas, "tas")
                    tas_yr = tas_yr.load() 
                    tas_yr.to_netcdf(outpath+f"{model}_{rea}_tas_yr.nc")

In [ ]:
# dictionary realiz for each model
cmip6_picontrol_tas_msftyz_dict = {}
for model in picontrol_tas_cmip6.keys():
    if model in cmip6_picontrol_amoc_msftyz_dict.keys():
        cmip6_picontrol_tas_msftyz_dict[model] = []
        for rea in picontrol_tas_cmip6[model].keys():
            if rea in cmip6_picontrol_amoc_msftyz_dict[model]:
                cmip6_picontrol_tas_msftyz_dict[model].append(rea)
        if not cmip6_picontrol_tas_msftyz_dict[model]: #remove key from dict if realiz list empty 
            del cmip6_picontrol_tas_msftyz_dict[model]

with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/picontrol/cmip6_picontrol_tas_msftyz_dict.json", 'w') as json_file:
    json.dump(cmip6_picontrol_tas_msftyz_dict, json_file)

seasonal data

In [ ]:
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/picontrol/cmip6_picontrol_amoc_msftyz_dict.json", 'r') as json_file:
    cmip6_picontrol_amoc_msftyz_dict = json.load(json_file)

In [ ]:
for model in picontrol_tas_cmip6.keys():
    if model in cmip6_picontrol_amoc_msftyz_dict.keys():
        for rea in picontrol_tas_cmip6[model].keys():
            if rea in cmip6_picontrol_amoc_msftyz_dict[model]:
                tas = picontrol_tas_cmip6[model][rea]
    
                outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/picontrol/{model}/tas_djf/"
                if not os.path.exists(outpath):
                    os.makedirs(outpath)
                tas_sea = tas.resample(time='QS-DEC').mean(dim='time')
                tas_djf = tas_sea.sel(time=tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1))
                tas_djf = reset_djf_time(tas_djf)
                new_time = xr.cftime_range(start='0001', periods=tas_djf.sizes['time'], freq='YS', calendar='proleptic_gregorian')
                tas_djf = tas_djf.assign_coords(time=new_time)
                if not os.path.isfile(outpath+f"{model}_{rea}_tas_djf.nc"):
                    tas_djf.to_netcdf(outpath+f"{model}_{rea}_tas_djf.nc")
                    
                outpath = f"/work/uo1075/m300817/teu_amoc/data/CMIP6/picontrol/{model}/tas_jja/"
                if not os.path.exists(outpath):
                    os.makedirs(outpath)
                tas_sea = tas.resample(time='QS').mean(dim='time')
                tas_jja = tas_sea.sel(time=tas_sea.time.dt.season=='JJA')
                new_time = xr.cftime_range(start='0001', periods=tas_jja.sizes['time'], freq='YS', calendar='proleptic_gregorian')
                tas_jja = tas_jja.assign_coords(time=new_time)
                if not os.path.isfile(outpath+f"{model}_{rea}_tas_jja.nc"):
                    tas_jja.to_netcdf(outpath+f"{model}_{rea}_tas_jja.nc")

# NAHosMIP

## AMOC

CanESM5

In [ ]:
model = "CanESM5"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
msft = xr.open_mfdataset(path+f"/{model}/u03-hos/*msft*.nc", use_cftime=True, parallel=True)

amoc_yr = weighted_mon_to_year_mean(msft.sel(x=0, drop=True).sel(lat=26.5, method='nearest', drop=True), "msftmz")
max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftmz
amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
amoc_yr = amoc_yr.load()
new_time = xr.cftime_range(start='0001', periods=amoc_yr.sizes['time'], freq='YS', calendar='proleptic_gregorian')
amoc_yr = amoc_yr.assign_coords(time=new_time)
amoc_yr.to_netcdf(path+f"post/{model}_u03-hos_amoc26_yr.nc")

CESM2

In [ ]:
model = "CESM2"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
msft = xr.open_mfdataset(path+f"/{model}/u03-hos/*msft*.nc", use_cftime=True, parallel=True)

amoc_yr = msft.MOC.sel(lat_aux_grid=26.5, method='nearest', drop=True).sel(moc_comp=0).sel(transport_reg=1)
max_moc_z_idx= amoc_yr.idxmax(dim='moc_z')
amoc_yr = (amoc_yr.sel(moc_z=max_moc_z_idx, drop=True))
amoc_yr = amoc_yr.load()
new_time = xr.cftime_range(start='0001', periods=amoc_yr.sizes['time'], freq='YS', calendar='proleptic_gregorian')
amoc_yr = amoc_yr.assign_coords(time=new_time)
amoc_yr = amoc_yr.isel(time=slice(0, -28)) # last few years are anyways empty and last year only partly run
amoc_yr.to_netcdf(path+f"post/{model}_u03-hos_amoc26_yr.nc")

EC-Earth3

In [ ]:
model = "EC-Earth3"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
msft = xr.open_mfdataset(path+f"/{model}/u03-hos/*msft*.nc", use_cftime=True, parallel=True)

amoc_yr = msft.sel(rlat=26.5, method='nearest', drop=True).sel(basin=1).msftyz
max_lev_idx= amoc_yr.idxmax(dim='lev') #select depths with maximum msftmz
amoc_yr = (amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)
amoc_yr = amoc_yr.load()
new_time = xr.cftime_range(start='0001', periods=amoc_yr.sizes['time'], freq='YS', calendar='proleptic_gregorian')
amoc_yr = amoc_yr.assign_coords(time=new_time)
amoc_yr.to_netcdf(path+f"post/{model}_u03-hos_amoc26_yr.nc")

HadGEM3-GC3-1LL

In [ ]:
model = "HadGEM3-GC3-1LL"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
msft = xr.open_mfdataset(path+f"/{model}/u03-hos/*msft*.nc", use_cftime=True, parallel=True)

amoc_yr = msft.zomsfatl
max_depthw_idx= amoc_yr.idxmax(dim='depthw') #select depths with maximum msftmz
amoc_yr = (amoc_yr.sel(depthw=max_depthw_idx, drop=True))
for y in amoc_yr.y.values:
    if amoc_yr.sel(y=y).nav_lat == 26.053894:
        amoc_yr = amoc_yr.sel(y=y)
        break
amoc_yr = amoc_yr.load()
new_time = xr.cftime_range(start='0001', periods=amoc_yr.sizes['time_counter'], freq='YS', calendar='proleptic_gregorian')
amoc_yr = amoc_yr.assign_coords(time_counter=new_time).rename({'time_counter':'time'}).isel(x=0)
amoc_yr.to_netcdf(path+f"post/{model}_u03-hos_amoc26_yr.nc")

HadGEM3-GC3-1MM

In [ ]:
model = "HadGEM3-GC3-1MM"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
msft = xr.open_mfdataset(path+f"/{model}/u03-hos/*msft*.nc", use_cftime=True, parallel=True)

amoc_yr = msft.zomsfatl
max_depthw_idx= amoc_yr.idxmax(dim='depthw') #select depths with maximum msftmz
amoc_yr = (amoc_yr.sel(depthw=max_depthw_idx, drop=True))
for y in amoc_yr.y.values:
    if amoc_yr.sel(y=y).nav_lat == 26.502243:
        amoc_yr = amoc_yr.sel(y=y)
        break
amoc_yr = amoc_yr.load()
new_time = xr.cftime_range(start='0001', periods=amoc_yr.sizes['time_counter'], freq='YS', calendar='proleptic_gregorian')
amoc_yr = amoc_yr.assign_coords(time_counter=new_time).rename({'time_counter':'time'}).isel(x=0)
amoc_yr.to_netcdf(path+f"post/{model}_u03-hos_amoc26_yr.nc")

IPSL-CM6A-LR

In [ ]:
model = "IPSL-CM6A-LR"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
msft = xr.open_mfdataset(path+f"/{model}/u03-hos/*msft*.nc", use_cftime=True, parallel=True)

amoc_yr = msft.zomsfatl
for y in amoc_yr.y.values:
    if amoc_yr.sel(y=y).nav_lat == 26.056036:
        amoc_yr = amoc_yr.sel(y=y)
        break
max_olevel_idx= amoc_yr.idxmax(dim='olevel') #select depths with maximum msftmz
amoc_yr = (amoc_yr.sel(olevel=max_olevel_idx, drop=True))
amoc_yr = amoc_yr.load()
new_time = xr.cftime_range(start='0001', periods=amoc_yr.sizes['time_counter'], freq='YS', calendar='proleptic_gregorian')
amoc_yr = amoc_yr.assign_coords(time_counter=new_time).rename({'time_counter':'time'}).isel(x=0)
amoc_yr.to_netcdf(path+f"post/{model}_u03-hos_amoc26_yr.nc")

MPI-ESM1-2-LR

In [ ]:
model = "MPI-ESM1-2-LR"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
msft = xr.open_mfdataset(path+f"/{model}/u03-hos/*msft*.nc", use_cftime=True, parallel=True)

atlmoc = msft.drop_vars([var for var in msft.data_vars if var != "atlantic_moc"])
amoc_yr = (atlmoc.sel(lat=26.5, drop=True).sel(lon=0, drop=True)/(1025.0 * 10**6)).compute()
max_dep_idx= amoc_yr.idxmax(dim='depth_2')
amoc_yr = amoc_yr.sel(depth_2=max_dep_idx.atlantic_moc, drop=True).atlantic_moc
new_time = xr.cftime_range(start='0001', periods=amoc_yr.sizes['time'], freq='YS', calendar='proleptic_gregorian')
amoc_yr = amoc_yr.assign_coords(time=new_time)
amoc_yr.to_netcdf(path+f"post/{model}_u03-hos_amoc26_yr.nc")

MPI-ESM1-2-HR

In [ ]:
model = "MPI-ESM1-2-HR"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
msft = xr.open_mfdataset(path+f"/{model}/u03-hos/*msft*.nc", use_cftime=True, parallel=True)

atlmoc = msft.drop_vars([var for var in msft.data_vars if var != "atlantic_moc"])
amoc_yr = (atlmoc.sel(lat=26.5, drop=True).sel(lon=0, drop=True)/(1025.0 * 10**6)).compute()
max_dep_idx= amoc_yr.idxmax(dim='depth_2')
amoc_yr = amoc_yr.sel(depth_2=max_dep_idx.atlantic_moc, drop=True).atlantic_moc
new_time = xr.cftime_range(start='0001', periods=amoc_yr.sizes['time'], freq='YS', calendar='proleptic_gregorian')
amoc_yr = amoc_yr.assign_coords(time=new_time)
amoc_yr.to_netcdf(path+f"post/{model}_u03-hos_amoc26_yr.nc")

## SAT

CanESM5

In [ ]:
# Annual mean T
model = "CanESM5"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
tas = xr.open_mfdataset(path+f"/{model}/u03-hos/*tas*.nc", use_cftime=True, parallel=True)
tas = weighted_mon_to_year_mean(tas, "tas")
new_time = xr.cftime_range(start='0001', periods=tas.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas = tas.assign_coords(time=new_time)

mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(tas)  # Europe
mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(tas) # Natural earth masks
tas_eur = tas.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)

tas.to_netcdf(path+f"post/{model}_u03-hos_tas_yr.nc")
tas_eur.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_yr.nc")

In [ ]:
# Winter & summer mean T
model = "CanESM5"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
tas_mon = xr.open_mfdataset(path+f"/{model}/u03-hos/*tas*.nc", use_cftime=True, parallel=True)

# Winter
tas_sea = tas_mon.resample(time='QS-DEC').mean(dim='time')
tas_djf = tas_sea.sel(time=tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1))
tas_djf = reset_djf_time(tas_djf)
new_time = xr.cftime_range(start='0001', periods=tas_djf.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas_djf = tas_djf.assign_coords(time=new_time)
mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(tas_djf)  # Europe
mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(tas_djf) # Natural earth masks
tas_eur_djf = tas_djf.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
tas_djf.to_netcdf(path+f"post/{model}_u03-hos_tas_djf.nc")
tas_eur_djf.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_djf.nc")

# Summer
tas_sea = tas_mon.resample(time='QS').mean(dim='time')
tas_jja = tas_sea.sel(time=tas_sea.time.dt.season=='JJA')
new_time = xr.cftime_range(start='0001', periods=tas_jja.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas_jja = tas_jja.assign_coords(time=new_time)
tas_eur_jja = tas_jja.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
tas_jja.to_netcdf(path+f"post/{model}_u03-hos_tas_jja.nc")
tas_eur_jja.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_jja.nc")

CESM2

In [ ]:
model = "CESM2"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
tas = xr.open_mfdataset(path+f"/{model}/u03-hos/*tas*.nc", use_cftime=True, parallel=True)
tas = weighted_mon_to_year_mean(tas, "TREFHT")
new_time = xr.cftime_range(start='0001', periods=tas.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas = tas.assign_coords(time=new_time)
tas = tas.isel(time=slice(0, -28)) # last few years are anyways empty and last year only partly run

mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(tas)  # Europe
mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(tas) # Natural earth masks
tas_eur = tas.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)

tas.to_netcdf(path+f"post/{model}_u03-hos_tas_yr.nc")
tas_eur.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_yr.nc")

In [ ]:
# Winter & summer mean T
model = "CESM2"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
tas_mon = xr.open_mfdataset(path+f"/{model}/u03-hos/*tas*.nc", use_cftime=True, parallel=True).rename({'TREFHT': 'tas'})

# Winter
tas_sea = tas_mon.resample(time='QS-DEC').mean(dim='time')
tas_djf = tas_sea.sel(time=tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1))
tas_djf = reset_djf_time(tas_djf)
new_time = xr.cftime_range(start='0001', periods=tas_djf.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas_djf = tas_djf.assign_coords(time=new_time)
mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(tas_djf)  # Europe
mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(tas_djf) # Natural earth masks
tas_eur_djf = tas_djf.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
tas_djf.to_netcdf(path+f"post/{model}_u03-hos_tas_djf.nc")
tas_eur_djf.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_djf.nc")

# Summer
tas_sea = tas_mon.resample(time='QS').mean(dim='time')
tas_jja = tas_sea.sel(time=tas_sea.time.dt.season=='JJA')
new_time = xr.cftime_range(start='0001', periods=tas_jja.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas_jja = tas_jja.assign_coords(time=new_time)
tas_eur_jja = tas_jja.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
tas_jja.to_netcdf(path+f"post/{model}_u03-hos_tas_jja.nc")
tas_eur_jja.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_jja.nc")

EC-Earth3

In [ ]:
model = "EC-Earth3"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
tas = xr.open_mfdataset(path+f"/{model}/u03-hos/*tas*.nc", use_cftime=True, parallel=True)
tas = weighted_mon_to_year_mean(tas, "tas")
new_time = xr.cftime_range(start='0001', periods=tas.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas = tas.assign_coords(time=new_time)

mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(tas)  # Europe
mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(tas) # Natural earth masks
tas_eur = tas.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)

tas.to_netcdf(path+f"post/{model}_u03-hos_tas_yr.nc")
tas_eur.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_yr.nc")

In [ ]:
# Winter & summer mean T
model = "EC-Earth3"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
tas_mon = xr.open_mfdataset(path+f"/{model}/u03-hos/*tas*.nc", use_cftime=True, parallel=True)

# Winter
tas_sea = tas_mon.resample(time='QS-DEC').mean(dim='time')
tas_djf = tas_sea.sel(time=tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1))
tas_djf = reset_djf_time(tas_djf)
new_time = xr.cftime_range(start='0001', periods=tas_djf.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas_djf = tas_djf.assign_coords(time=new_time)
mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(tas_djf)  # Europe
mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(tas_djf) # Natural earth masks
tas_eur_djf = tas_djf.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
tas_djf.to_netcdf(path+f"post/{model}_u03-hos_tas_djf.nc")
tas_eur_djf.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_djf.nc")

# Summer
tas_sea = tas_mon.resample(time='QS').mean(dim='time')
tas_jja = tas_sea.sel(time=tas_sea.time.dt.season=='JJA')
new_time = xr.cftime_range(start='0001', periods=tas_jja.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas_jja = tas_jja.assign_coords(time=new_time)
tas_eur_jja = tas_jja.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
tas_jja.to_netcdf(path+f"post/{model}_u03-hos_tas_jja.nc")
tas_eur_jja.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_jja.nc")

HadGEM3-GC3-1LL

In [ ]:
model = "HadGEM3-GC3-1LL"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
tas = xr.open_mfdataset(path+f"/{model}/u03-hos/*tas*.nc", use_cftime=True, parallel=True)
tas = weighted_mon_to_year_mean(tas, "tas")
new_time = xr.cftime_range(start='0001', periods=tas.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas = tas.assign_coords(time=new_time)

mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(tas)  # Europe
mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(tas) # Natural earth masks

tas_eur = tas.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
tas.to_netcdf(path+f"post/{model}_u03-hos_tas_yr.nc")
tas_eur.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_yr.nc")

In [ ]:
# Winter & summer mean T
model = "HadGEM3-GC3-1LL"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
tas_mon = xr.open_mfdataset(path+f"/{model}/u03-hos/*tas*.nc", use_cftime=True, parallel=True)

# Winter
tas_sea = tas_mon.resample(time='QS-DEC').mean(dim='time')
tas_djf = tas_sea.sel(time=tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1))
tas_djf = reset_djf_time(tas_djf)
new_time = xr.cftime_range(start='0001', periods=tas_djf.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas_djf = tas_djf.assign_coords(time=new_time)
mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(tas_djf)  # Europe
mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(tas_djf) # Natural earth masks
tas_eur_djf = tas_djf.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
tas_djf.to_netcdf(path+f"post/{model}_u03-hos_tas_djf.nc")
tas_eur_djf.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_djf.nc")

# Summer
tas_sea = tas_mon.resample(time='QS').mean(dim='time')
tas_jja = tas_sea.sel(time=tas_sea.time.dt.season=='JJA')
new_time = xr.cftime_range(start='0001', periods=tas_jja.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas_jja = tas_jja.assign_coords(time=new_time)
tas_eur_jja = tas_jja.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
tas_jja.to_netcdf(path+f"post/{model}_u03-hos_tas_jja.nc")
tas_eur_jja.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_jja.nc")

HadGEM3-GC3-1MM

In [ ]:
model = "HadGEM3-GC3-1MM"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
tas = xr.open_mfdataset(path+f"/{model}/u03-hos/*tas*.nc", use_cftime=True, parallel=True)
tas = weighted_mon_to_year_mean(tas, "tas")
new_time = xr.cftime_range(start='0001', periods=tas.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas = tas.assign_coords(time=new_time)

mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(tas)  # Europe
mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(tas) # Natural earth masks

tas_eur = tas.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
tas.to_netcdf(path+f"post/{model}_u03-hos_tas_yr.nc")
tas_eur.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_yr.nc")

In [ ]:
# Winter & summer mean T
model = "HadGEM3-GC3-1MM"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
tas_mon = xr.open_mfdataset(path+f"/{model}/u03-hos/*tas*.nc", use_cftime=True, parallel=True)

# Winter
tas_sea = tas_mon.resample(time='QS-DEC').mean(dim='time')
tas_djf = tas_sea.sel(time=tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1))
tas_djf = reset_djf_time(tas_djf)
new_time = xr.cftime_range(start='0001', periods=tas_djf.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas_djf = tas_djf.assign_coords(time=new_time)
mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(tas_djf)  # Europe
mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(tas_djf) # Natural earth masks
tas_eur_djf = tas_djf.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
tas_djf.to_netcdf(path+f"post/{model}_u03-hos_tas_djf.nc")
tas_eur_djf.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_djf.nc")

# Summer
tas_sea = tas_mon.resample(time='QS').mean(dim='time')
tas_jja = tas_sea.sel(time=tas_sea.time.dt.season=='JJA')
new_time = xr.cftime_range(start='0001', periods=tas_jja.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas_jja = tas_jja.assign_coords(time=new_time)
tas_eur_jja = tas_jja.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
tas_jja.to_netcdf(path+f"post/{model}_u03-hos_tas_jja.nc")
tas_eur_jja.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_jja.nc")

IPSL-CM6A-LR

In [ ]:
model = "IPSL-CM6A-LR"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
tas = xr.open_mfdataset(path+f"/{model}/u03-hos/*tas*.nc", use_cftime=True, parallel=True)
tas = weighted_mon_to_year_mean(tas, "tas")
new_time = xr.cftime_range(start='0001', periods=tas.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas = tas.assign_coords(time=new_time)

mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(tas)  # Europe
mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(tas) # Natural earth masks

tas_eur = tas.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
tas.to_netcdf(path+f"post/{model}_u03-hos_tas_yr.nc")
tas_eur.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_yr.nc")

In [ ]:
# Winter & summer mean T
model = "IPSL-CM6A-LR"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
tas_mon = xr.open_mfdataset(path+f"/{model}/u03-hos/*tas*.nc", use_cftime=True, parallel=True)

# Winter
tas_sea = tas_mon.resample(time='QS-DEC').mean(dim='time')
tas_djf = tas_sea.sel(time=tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1))
tas_djf = reset_djf_time(tas_djf)
new_time = xr.cftime_range(start='0001', periods=tas_djf.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas_djf = tas_djf.assign_coords(time=new_time)
mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(tas_djf)  # Europe
mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(tas_djf) # Natural earth masks
tas_eur_djf = tas_djf.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
tas_djf.to_netcdf(path+f"post/{model}_u03-hos_tas_djf.nc")
tas_eur_djf.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_djf.nc")

# Summer
tas_sea = tas_mon.resample(time='QS').mean(dim='time')
tas_jja = tas_sea.sel(time=tas_sea.time.dt.season=='JJA')
new_time = xr.cftime_range(start='0001', periods=tas_jja.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas_jja = tas_jja.assign_coords(time=new_time)
tas_eur_jja = tas_jja.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
tas_jja.to_netcdf(path+f"post/{model}_u03-hos_tas_jja.nc")
tas_eur_jja.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_jja.nc")

MPI-ESM1-2-LR

In [ ]:
model = "MPI-ESM1-2-LR"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
tas = xr.open_mfdataset(path+f"/{model}/u03-hos/*tas*.nc", use_cftime=True, parallel=True)
tas = tas.assign_coords(time=xr.cftime_range(start="1850-01-01", end="2049-12-31", freq="M", calendar="proleptic_gregorian"))
tas = weighted_mon_to_year_mean(tas, "tas")
new_time = xr.cftime_range(start='0001', periods=tas.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas = tas.assign_coords(time=new_time)

mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(tas)  # Europe
mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(tas) # Natural earth masks

tas_eur = tas.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
tas.to_netcdf(path+f"post/{model}_u03-hos_tas_yr.nc")
tas_eur.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_yr.nc")

In [ ]:
# Winter & summer mean T
model = "MPI-ESM1-2-LR"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
tas_mon = xr.open_mfdataset(path+f"/{model}/u03-hos/*tas*.nc", use_cftime=True, parallel=True)
tas_mon = tas_mon.assign_coords(time=xr.cftime_range(start="1850-01-01", end="2049-12-31", freq="M", calendar="proleptic_gregorian"))

# Winter
tas_sea = tas_mon.resample(time='QS-DEC').mean(dim='time')
tas_djf = tas_sea.sel(time=tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1))
tas_djf = reset_djf_time(tas_djf)
new_time = xr.cftime_range(start='0001', periods=tas_djf.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas_djf = tas_djf.assign_coords(time=new_time)
mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(tas_djf)  # Europe
mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(tas_djf) # Natural earth masks
tas_eur_djf = tas_djf.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
tas_djf.to_netcdf(path+f"post/{model}_u03-hos_tas_djf.nc")
tas_eur_djf.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_djf.nc")

# Summer
tas_sea = tas_mon.resample(time='QS').mean(dim='time')
tas_jja = tas_sea.sel(time=tas_sea.time.dt.season=='JJA')
new_time = xr.cftime_range(start='0001', periods=tas_jja.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas_jja = tas_jja.assign_coords(time=new_time)
tas_eur_jja = tas_jja.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
tas_jja.to_netcdf(path+f"post/{model}_u03-hos_tas_jja.nc")
tas_eur_jja.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_jja.nc")

MPI-ESM1-2-HR

In [ ]:
model = "MPI-ESM1-2-HR"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
tas = xr.open_mfdataset(path+f"/{model}/u03-hos/*tas*.nc", use_cftime=True, parallel=True)
tas = tas.assign_coords(time=xr.cftime_range(start="1850-01-01", end="1999-12-31", freq="M", calendar="proleptic_gregorian"))
tas = weighted_mon_to_year_mean(tas, "tas")
new_time = xr.cftime_range(start='0001', periods=tas.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas = tas.assign_coords(time=new_time)

mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(tas)  # Europe
mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(tas) # Natural earth masks

tas_eur = tas.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
tas.to_netcdf(path+f"post/{model}_u03-hos_tas_yr.nc")
tas_eur.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_yr.nc")

In [ ]:
# Winter & summer mean T
model = "MPI-ESM1-2-HR"
path = "/work/uo1075/m300817/hosing/nahosmip/" 
tas_mon = xr.open_mfdataset(path+f"/{model}/u03-hos/*tas*.nc", use_cftime=True, parallel=True)
tas_mon = tas_mon.assign_coords(time=xr.cftime_range(start="1850-01-01", end="1999-12-31", freq="M", calendar="proleptic_gregorian"))

# Winter
tas_sea = tas_mon.resample(time='QS-DEC').mean(dim='time')
tas_djf = tas_sea.sel(time=tas_sea.time.dt.season=='DJF').isel(time=slice(None, -1))
tas_djf = reset_djf_time(tas_djf)
new_time = xr.cftime_range(start='0001', periods=tas_djf.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas_djf = tas_djf.assign_coords(time=new_time)
mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(tas_djf)  # Europe
mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(tas_djf) # Natural earth masks
tas_eur_djf = tas_djf.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
tas_djf.to_netcdf(path+f"post/{model}_u03-hos_tas_djf.nc")
tas_eur_djf.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_djf.nc")

# Summer
tas_sea = tas_mon.resample(time='QS').mean(dim='time')
tas_jja = tas_sea.sel(time=tas_sea.time.dt.season=='JJA')
new_time = xr.cftime_range(start='0001', periods=tas_jja.sizes['time'], freq='YS', calendar='proleptic_gregorian')
tas_jja = tas_jja.assign_coords(time=new_time)
tas_eur_jja = tas_jja.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
tas_jja.to_netcdf(path+f"post/{model}_u03-hos_tas_jja.nc")
tas_eur_jja.to_netcdf(path+f"post/{model}_u03-hos_tas_eur_jja.nc")

In [ ]:
moc_hos585 = xr.open_dataset("/work/uo1075/m300817/teu_amoc/data/CESM2_hosing/HOS_585/MOC_yr_hosing_05_2015_2100_0585.nc")

In [ ]:
for exp in ['CTL_126', 'CTL_585', 'HOS_126', 'HOS_585']:
    path = f"/work/uo1075/m300817/teu_amoc/data/CESM2_hosing/{exp}/"
    moc = xr.open_mfdataset(path+"MOC*.nc")
    amoc = moc.sel(lat_aux_grid=26.5, method='nearest').sel(transport_reg=1).sel(moc_comp=0).MOC
    max_idx = amoc.idxmax(dim='moc_z')
    amoc = amoc.sel(moc_z=max_idx)
    amoc.to_netcdf(path+f"{exp}_amoc26_yr.nc")

In [ ]:
path = f"/work/uo1075/m300817/teu_amoc/data/CESM2_hosing/CTL_126/"
tas = xr.open_mfdataset(path+"TEMP_*2016*.nc")
tas.TEMP